In [141]:
# ============================================================
# R4-01 — CLEAN COLAB RECOVERY
# Uncertainty-Aware Selective Escalation and Robustness
# ============================================================

import os
import shutil
import zipfile
import hashlib
from pathlib import Path

print("=" * 100)
print("R4-01 — CLEAN COLAB RECOVERY")
print("=" * 100)

# Clean only the new R4 workspace
R4_ROOT = Path("/content/R4_work")

if R4_ROOT.exists():
    shutil.rmtree(R4_ROOT)

R4_ROOT.mkdir(parents=True, exist_ok=True)

print("R4 workspace:", R4_ROOT)
print("Workspace ready.")

R4-01 — CLEAN COLAB RECOVERY
R4 workspace: /content/R4_work
Workspace ready.


In [142]:
from google.colab import files

print("=" * 100)
print("UPLOAD FINAL FROZEN R3 ZIP")
print("=" * 100)

uploaded = files.upload()

print("\nUploaded files:")
for name in uploaded:
    print(" ", name)

UPLOAD FINAL FROZEN R3 ZIP


Saving XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip to XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip

Uploaded files:
  XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip


In [143]:
from pathlib import Path
import hashlib

print("=" * 100)
print("R4 — VERIFY DUPLICATE R3 ZIP FILES")
print("=" * 100)

files_to_check = [
    Path("/content/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip"),
    Path("/content/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip"),
]

hashes = {}

for f in files_to_check:
    if not f.exists():
        print(f"\nMISSING: {f.name}")
        continue

    sha = hashlib.sha256(f.read_bytes()).hexdigest()
    hashes[f.name] = sha

    print(f"\nFile   : {f.name}")
    print(f"Size   : {f.stat().st_size:,} bytes")
    print(f"SHA256 : {sha}")

print("\n" + "-" * 100)

if len(hashes) == 2:
    values = list(hashes.values())

    if values[0] == values[1]:
        print("PASS — Both ZIP files are byte-for-byte identical.")
        print("Safe to keep the newly uploaded (1).zip as the R4 input.")
    else:
        print("STOP — ZIP files are NOT identical.")
        print("Do not continue with R4.")
else:
    print("STOP — Could not locate both ZIP files.")

R4 — VERIFY DUPLICATE R3 ZIP FILES

File   : XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip
Size   : 329,177 bytes
SHA256 : ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152

File   : XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip
Size   : 329,177 bytes
SHA256 : ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152

----------------------------------------------------------------------------------------------------
PASS — Both ZIP files are byte-for-byte identical.
Safe to keep the newly uploaded (1).zip as the R4 input.


In [144]:
# ====================================================================================================
# R4-02 — EXTRACT VERIFIED FROZEN R3 PACKAGE
# ====================================================================================================

from pathlib import Path
import zipfile
import shutil
import hashlib

print("=" * 100)
print("R4-02 — EXTRACT VERIFIED FROZEN R3 PACKAGE")
print("=" * 100)

INPUT_ZIP = Path(
    "/content/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip"
)

EXPECTED_SHA256 = (
    "ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152"
)

R4_ROOT = Path("/content/R4_work")
RECOVERED = R4_ROOT / "recovered_R3"

# --------------------------------------------------------------------------------
# Verify input ZIP
# --------------------------------------------------------------------------------

assert INPUT_ZIP.exists(), f"STOP — ZIP not found: {INPUT_ZIP}"

actual_sha = hashlib.sha256(INPUT_ZIP.read_bytes()).hexdigest()

print(f"\nInput ZIP : {INPUT_ZIP.name}")
print(f"Size      : {INPUT_ZIP.stat().st_size:,} bytes")
print(f"SHA256    : {actual_sha}")

assert actual_sha == EXPECTED_SHA256, (
    "STOP — Input ZIP SHA256 does not match verified R3 package."
)

print("PASS — frozen R3 ZIP SHA256 verified")

# --------------------------------------------------------------------------------
# Prepare clean recovery directory
# --------------------------------------------------------------------------------

if RECOVERED.exists():
    shutil.rmtree(RECOVERED)

RECOVERED.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------------------------------------
# Test ZIP integrity before extraction
# --------------------------------------------------------------------------------

with zipfile.ZipFile(INPUT_ZIP, "r") as zf:

    bad_file = zf.testzip()

    if bad_file is not None:
        raise RuntimeError(
            f"STOP — ZIP integrity failure at member: {bad_file}"
        )

    members = zf.namelist()

    print(f"PASS — ZIP integrity test")
    print(f"ZIP members: {len(members)}")

    zf.extractall(RECOVERED)

# --------------------------------------------------------------------------------
# Count recovered files
# --------------------------------------------------------------------------------

recovered_files = [
    p for p in RECOVERED.rglob("*")
    if p.is_file()
]

print(f"\nRecovered directory : {RECOVERED}")
print(f"Recovered files     : {len(recovered_files)}")

print("\nTop-level recovered contents")
print("-" * 100)

for p in sorted(RECOVERED.iterdir()):
    if p.is_dir():
        print(f"[DIR ] {p.name}")
    else:
        print(f"[FILE] {p.name}")

print("\nPASS — frozen R3 package recovered successfully")

R4-02 — EXTRACT VERIFIED FROZEN R3 PACKAGE

Input ZIP : XAgentic_R3_Candidate_Masking_FINAL_2026-10-07 (1).zip
Size      : 329,177 bytes
SHA256    : ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152
PASS — frozen R3 ZIP SHA256 verified
PASS — ZIP integrity test
ZIP members: 30

Recovered directory : /content/R4_work/recovered_R3
Recovered files     : 30

Top-level recovered contents
----------------------------------------------------------------------------------------------------
[DIR ] XAgentic_R3_Candidate_Masking_FINAL

PASS — frozen R3 package recovered successfully


In [145]:
# ====================================================================================================
# R4-03 — LOCATE FROZEN R3 EVIDENCE
# ====================================================================================================

from pathlib import Path

print("=" * 100)
print("R4-03 — LOCATE FROZEN R3 EVIDENCE")
print("=" * 100)

RECOVERED = Path("/content/R4_work/recovered_R3")

targets = [
    "R3_final_frozen_policy_heldout_decisions.csv",
    "R3_final_system_comparison.csv",
]

all_ok = True

for target in targets:

    found = list(RECOVERED.rglob(target))

    print(f"\nTARGET: {target}")
    print("-" * 100)

    if len(found) == 0:
        print("MISSING")
        all_ok = False

    elif len(found) == 1:
        p = found[0]
        print("PASS — found")
        print("Path:", p)
        print("Size:", f"{p.stat().st_size:,}", "bytes")

    else:
        print(f"WARNING — {len(found)} copies found")
        for p in found:
            print(" ", p)

print("\n" + "=" * 100)

if all_ok:
    print("PASS — critical frozen R3 evidence located")
else:
    print("STOP — required R3 evidence is missing")

R4-03 — LOCATE FROZEN R3 EVIDENCE

TARGET: R3_final_frozen_policy_heldout_decisions.csv
----------------------------------------------------------------------------------------------------
PASS — found
Path: /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/03_RESULTS_RAW/R3_final_frozen_policy_heldout_decisions.csv
Size: 86,890 bytes

TARGET: R3_final_system_comparison.csv
----------------------------------------------------------------------------------------------------
PASS — found
Path: /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/04_RESULTS_AGGREGATE/R3_final_system_comparison.csv
Size: 410 bytes

PASS — critical frozen R3 evidence located


In [146]:
# ====================================================================================================
# R4-04 — RECOVERED PACKAGE INVENTORY
# ====================================================================================================

from pathlib import Path

RECOVERED = Path("/content/R4_work/recovered_R3")

print("=" * 100)
print("R4-04 — RECOVERED R3 PACKAGE INVENTORY")
print("=" * 100)

files = sorted(
    [p for p in RECOVERED.rglob("*") if p.is_file()]
)

print(f"Total files: {len(files)}\n")

for p in files:
    rel = p.relative_to(RECOVERED)
    print(f"{str(rel):100s} {p.stat().st_size:>12,} bytes")

R4-04 — RECOVERED R3 PACKAGE INVENTORY
Total files: 30

XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv                        302 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_test_split.csv                             81,369 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen_oof_mass_policy.json                              866 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_feasible_mass_quantiles.csv                          229 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_training_calibration_rows.csv                    391,762 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction.json                         1,033 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction_cv.csv                         305 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_training_calibration_rows.csv                      

In [147]:
# ====================================================================================================
# R4-05 — IMMUTABLE R3 BASELINE INTEGRITY AUDIT
# ====================================================================================================

from pathlib import Path
import pandas as pd
import hashlib

print("=" * 100)
print("R4-05 — IMMUTABLE R3 BASELINE INTEGRITY AUDIT")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

MANIFEST = BASE / "06_FINAL_MANIFEST" / "R3_SHA256_MANIFEST.csv"

assert BASE.exists(), "STOP — recovered R3 base directory missing"
assert MANIFEST.exists(), "STOP — R3 SHA256 manifest missing"

manifest = pd.read_csv(MANIFEST)

print("\nManifest columns:")
print(list(manifest.columns))

print("\nManifest rows:", len(manifest))

print("\nFirst rows:")
print(manifest.head().to_string(index=False))

# Identify likely path/hash columns automatically
path_candidates = [
    c for c in manifest.columns
    if any(x in c.lower() for x in ["path", "file", "artifact"])
]

hash_candidates = [
    c for c in manifest.columns
    if "sha" in c.lower() or "hash" in c.lower()
]

print("\nPath-column candidates:", path_candidates)
print("Hash-column candidates:", hash_candidates)

assert path_candidates, "STOP — cannot identify manifest path column"
assert hash_candidates, "STOP — cannot identify manifest SHA256 column"

path_col = path_candidates[0]
hash_col = hash_candidates[0]

print("\nUsing:")
print(" path column :", path_col)
print(" hash column :", hash_col)

passed = 0
failed = 0
missing = 0

print("\n" + "-" * 100)
print("VERIFYING MANIFEST")
print("-" * 100)

for _, row in manifest.iterrows():

    rel = str(row[path_col]).strip()
    expected = str(row[hash_col]).strip().lower()

    # Manifest may use paths relative to package root.
    p = BASE / rel

    if not p.exists():
        print("MISSING:", rel)
        missing += 1
        continue

    actual = hashlib.sha256(p.read_bytes()).hexdigest()

    if actual == expected:
        passed += 1
    else:
        print("HASH FAIL:", rel)
        print(" expected:", expected)
        print(" actual  :", actual)
        failed += 1

print("\n" + "=" * 100)
print("R3 MANIFEST VERIFICATION RESULT")
print("=" * 100)

print("PASS   :", passed)
print("FAILED :", failed)
print("MISSING:", missing)

if failed == 0 and missing == 0:
    print("\nPASS — complete frozen R3 evidence package is intact.")
else:
    print("\nSTOP — R3 evidence integrity failure.")

R4-05 — IMMUTABLE R3 BASELINE INTEGRITY AUDIT

Manifest columns:
['relative_path', 'size_bytes', 'sha256']

Manifest rows: 29

First rows:
                                     relative_path  size_bytes                                                           sha256
02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv         302 0100ce08f0803fe8b41582deeb90a83695cac480e36aa28720c07f9e5ada9366
        02_PROVENANCE/R3_frozen1200_test_split.csv       81369 4dda072c138b3ba8b374e5b2025c6411ee11bfec9eb32d2c9c8556f096428a8a
      02_PROVENANCE/R3_frozen_oof_mass_policy.json         866 9ebab1f743c71ee4fb3202976a6d9846b03391e256fbc095bd561c5adf831774
  02_PROVENANCE/R3_oof_feasible_mass_quantiles.csv         229 d347d3f4350a59dca156f3ce63d921379f893d19924f96a717fa425c9c4df8c2
02_PROVENANCE/R3_oof_training_calibration_rows.csv      391762 2c6f8bb8b3f3e45b9f5e5a0dfa91140e3bfb2dd27061e929f479cad2354eaa16

Path-column candidates: ['relative_path']
Hash-column candidates: ['sha256']

Using:
 path c

In [148]:
# ====================================================================================================
# R4-06 — FROZEN R3 POLICY AUDIT
# ====================================================================================================

from pathlib import Path
import json
import pandas as pd

print("=" * 100)
print("R4-06 — FROZEN R3 POLICY AUDIT")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

summary_path = (
    BASE /
    "04_RESULTS_AGGREGATE" /
    "R3_final_frozen_policy_summary.json"
)

comparison_path = (
    BASE /
    "04_RESULTS_AGGREGATE" /
    "R3_final_system_comparison.csv"
)

decisions_path = (
    BASE /
    "03_RESULTS_RAW" /
    "R3_final_frozen_policy_heldout_decisions.csv"
)

with open(summary_path, "r") as f:
    policy = json.load(f)

comparison = pd.read_csv(comparison_path)
decisions = pd.read_csv(decisions_path)

print("\nFROZEN POLICY")
print("-" * 100)
print(json.dumps(policy, indent=2))

print("\nFINAL SYSTEM COMPARISON")
print("-" * 100)
print(comparison.to_string(index=False))

print("\nHELD-OUT DECISION TABLE")
print("-" * 100)
print("Rows   :", len(decisions))
print("Columns:", len(decisions.columns))

print("\nColumns:")
for i, c in enumerate(decisions.columns, 1):
    print(f"{i:2d}. {c}")

R4-06 — FROZEN R3 POLICY AUDIT

FROZEN POLICY
----------------------------------------------------------------------------------------------------
{
  "autonomous_accuracy": 0.9648093841642229,
  "autonomous_macro_f1": 0.9648195677002419,
  "autonomous_n": 341,
  "autonomous_strict_validity": 1.0,
  "correct_autonomous_n": 329,
  "correct_autonomous_over_total": 0.9138888888888889,
  "coverage": 0.9472222222222222,
  "empty_escalations": 4,
  "escalated_n": 19,
  "escalation_rate": 0.05277777777777778,
  "frozen_tau": 0.5486315987262682,
  "low_mass_escalations": 15,
  "n_total": 360,
  "policy_sha256": "9ebab1f743c71ee4fb3202976a6d9846b03391e256fbc095bd561c5adf831774",
  "remaining_correct_to_wrong": 0,
  "truth_feasible_escalated": 2,
  "truth_infeasible_escalated": 17
}

FINAL SYSTEM COMPARISON
----------------------------------------------------------------------------------------------------
                     system  coverage  escalation_rate  autonomous_accuracy  autonomous_st

In [149]:
# ====================================================================================================
# R4-07 — R4 UNCERTAINTY SIGNAL DISCOVERY
# ====================================================================================================

import pandas as pd
from pathlib import Path

print("=" * 100)
print("R4-07 — UNCERTAINTY SIGNAL DISCOVERY")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

files = {
    "heldout":
        BASE / "03_RESULTS_RAW" /
        "R3_final_frozen_policy_heldout_decisions.csv",

    "training_calibration":
        BASE / "02_PROVENANCE" /
        "R3_training_calibration_rows.csv",

    "oof_calibration":
        BASE / "02_PROVENANCE" /
        "R3_oof_training_calibration_rows.csv",

    "threshold_rows":
        BASE / "03_RESULTS_RAW" /
        "R3_feasible_mass_threshold_policy_rows.csv",
}

for name, path in files.items():

    print("\n" + "=" * 100)
    print(name.upper())
    print("=" * 100)

    df = pd.read_csv(path)

    print("File   :", path.name)
    print("Rows   :", len(df))
    print("Columns:", len(df.columns))

    print("\nCOLUMN LIST")

    for i, c in enumerate(df.columns, 1):
        print(f"{i:2d}. {c}")

    print("\nFIRST 3 ROWS")
    print(df.head(3).to_string(index=False))

R4-07 — UNCERTAINTY SIGNAL DISCOVERY

HELDOUT
File   : R3_final_frozen_policy_heldout_decisions.csv
Rows   : 360
Columns: 27

COLUMN LIST
 1. position
 2. id
 3. truth
 4. baseline_pred
 5. baseline_correct
 6. baseline_strict_valid
 7. strict_v2_feasible_set
 8. candidate_count
 9. p_eMBB
10. p_URLLC
11. p_mMTC
12. feasible_probability_mass
13. masked_p_eMBB
14. masked_p_URLLC
15. masked_p_mMTC
16. masked_pred
17. masked_correct
18. autonomous
19. action
20. transition
21. uc_kind
22. final_action
23. final_pred
24. final_escalation_reason
25. truth_strict_feasible
26. truth_strict_feasible_final
27. final_transition

FIRST 3 ROWS
 position  id truth baseline_pred  baseline_correct  baseline_strict_valid strict_v2_feasible_set  candidate_count   p_eMBB  p_URLLC   p_mMTC  feasible_probability_mass  masked_p_eMBB  masked_p_URLLC  masked_p_mMTC masked_pred masked_correct  autonomous      action      transition uc_kind final_action final_pred final_escalation_reason  truth_strict_feasible

In [150]:
# ====================================================================================================
# R4-08 — BUILD OOF UNCERTAINTY DEVELOPMENT TABLE
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("R4-08 — BUILD OOF UNCERTAINTY DEVELOPMENT TABLE")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

R4_OUT = Path("/content/R4_work/R4_results")
R4_OUT.mkdir(parents=True, exist_ok=True)

src = (
    BASE /
    "02_PROVENANCE" /
    "R3_oof_training_calibration_rows.csv"
)

df = pd.read_csv(src)

assert len(df) == 840
assert df["train_id"].nunique() == 840

prob_cols = [
    "oof_p_eMBB",
    "oof_p_URLLC",
    "oof_p_mMTC"
]

assert all(c in df.columns for c in prob_cols)

P = df[prob_cols].to_numpy(float)

# --------------------------------------------------------------------------------
# Probability integrity
# --------------------------------------------------------------------------------

row_sums = P.sum(axis=1)

print("\nProbability integrity")
print("-" * 100)
print("Minimum probability :", P.min())
print("Maximum probability :", P.max())
print("Min row sum         :", row_sums.min())
print("Max row sum         :", row_sums.max())

assert np.all(P >= 0)
assert np.all(P <= 1)
assert np.allclose(row_sums, 1.0, atol=1e-6)

print("PASS — OOF probability vectors are valid")

# --------------------------------------------------------------------------------
# Uncertainty signals
# --------------------------------------------------------------------------------

sorted_p = np.sort(P, axis=1)

df["r4_top1_confidence"] = sorted_p[:, -1]
df["r4_top2_confidence"] = sorted_p[:, -2]

df["r4_margin"] = (
    df["r4_top1_confidence"] -
    df["r4_top2_confidence"]
)

eps = 1e-15

entropy = -np.sum(
    P * np.log(np.clip(P, eps, 1.0)),
    axis=1
)

df["r4_entropy"] = entropy

# Normalize entropy to [0,1].
df["r4_normalized_entropy"] = (
    entropy / np.log(P.shape[1])
)

# --------------------------------------------------------------------------------
# Error / feasibility labels for DEVELOPMENT analysis only
# --------------------------------------------------------------------------------

df["r4_oof_error"] = ~df["oof_correct"].astype(bool)

df["r4_oof_strict_invalid"] = (
    ~df["oof_strict_valid"].astype(bool)
)

# Preserve R3 feasible mass signal
df["r4_feasible_mass"] = (
    df["oof_feasible_probability_mass"].astype(float)
)

# --------------------------------------------------------------------------------
# Save
# --------------------------------------------------------------------------------

keep = [
    "position",
    "train_id",
    "truth",
    "uc_kind",
    "ambiguous",
    "oof_fold",
    "oof_pred",
    "oof_correct",
    "oof_strict_valid",
    "candidate_count_oof",
    "oof_feasible_probability_mass",
    "oof_p_eMBB",
    "oof_p_URLLC",
    "oof_p_mMTC",
    "r4_top1_confidence",
    "r4_top2_confidence",
    "r4_margin",
    "r4_entropy",
    "r4_normalized_entropy",
    "r4_feasible_mass",
    "r4_oof_error",
    "r4_oof_strict_invalid",
]

out = df[keep].copy()

out_path = R4_OUT / "R4_oof_uncertainty_development.csv"
out.to_csv(out_path, index=False)

print("\nDevelopment table")
print("-" * 100)
print("Rows        :", len(out))
print("Errors      :", int(out["r4_oof_error"].sum()))
print("Correct     :", int((~out["r4_oof_error"]).sum()))
print("Strict bad  :", int(out["r4_oof_strict_invalid"].sum()))
print("Output      :", out_path)

print("\nUncertainty summary")
print("-" * 100)

summary_cols = [
    "r4_top1_confidence",
    "r4_margin",
    "r4_normalized_entropy",
    "r4_feasible_mass",
]

print(
    out[summary_cols]
    .describe(percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99])
    .T
    .to_string()
)

print("\nMean uncertainty by correctness")
print("-" * 100)

print(
    out.groupby("r4_oof_error")[summary_cols]
       .mean()
       .to_string()
)

print("\nPASS — R4 OOF uncertainty development table created")

R4-08 — BUILD OOF UNCERTAINTY DEVELOPMENT TABLE

Probability integrity
----------------------------------------------------------------------------------------------------
Minimum probability : 0.0001021150741858
Maximum probability : 0.9994258796005364
Min row sum         : 0.9999999999999996
Max row sum         : 1.0000000000000004
PASS — OOF probability vectors are valid

Development table
----------------------------------------------------------------------------------------------------
Rows        : 840
Errors      : 62
Correct     : 778
Strict bad  : 52
Output      : /content/R4_work/R4_results/R4_oof_uncertainty_development.csv

Uncertainty summary
----------------------------------------------------------------------------------------------------
                       count      mean       std       min        1%        5%       10%       25%       50%       75%       90%       95%       99%       max
r4_top1_confidence     840.0  0.942146  0.113228  0.409304  0.499338  0.668

In [151]:
# ====================================================================================================
# R4-09 — UNCERTAINTY ERROR-SEPARATION AUDIT
# ====================================================================================================

from sklearn.metrics import roc_auc_score
import pandas as pd
from pathlib import Path

print("=" * 100)
print("R4-09 — UNCERTAINTY ERROR-SEPARATION AUDIT")
print("=" * 100)

path = Path(
    "/content/R4_work/R4_results/"
    "R4_oof_uncertainty_development.csv"
)

df = pd.read_csv(path)

y = df["r4_oof_error"].astype(int)

signals = {
    "1 - top1 confidence":
        1.0 - df["r4_top1_confidence"],

    "1 - top1/top2 margin":
        1.0 - df["r4_margin"],

    "normalized entropy":
        df["r4_normalized_entropy"],

    "1 - feasible probability mass":
        1.0 - df["r4_feasible_mass"],
}

print("\nOOF errors:", int(y.sum()))
print("OOF correct:", int((1-y).sum()))

rows = []

for name, score in signals.items():

    auc = roc_auc_score(y, score)

    rows.append({
        "uncertainty_signal": name,
        "error_detection_auc": auc
    })

result = pd.DataFrame(rows).sort_values(
    "error_detection_auc",
    ascending=False
)

print("\nError-detection discrimination")
print("-" * 100)
print(result.to_string(index=False))

result.to_csv(
    "/content/R4_work/R4_results/"
    "R4_uncertainty_signal_auc.csv",
    index=False
)

# --------------------------------------------------------------------------------
# Quantile error concentration
# --------------------------------------------------------------------------------

print("\nHighest-uncertainty error concentration")
print("-" * 100)

for name, score in signals.items():

    tmp = df.copy()
    tmp["_score"] = score

    tmp = tmp.sort_values(
        "_score",
        ascending=False
    ).reset_index(drop=True)

    print(f"\n{name}")

    for frac in [0.05, 0.10, 0.15, 0.20]:

        n = max(1, int(round(len(tmp) * frac)))

        subset = tmp.iloc[:n]

        errors_captured = int(
            subset["r4_oof_error"].sum()
        )

        total_errors = int(
            tmp["r4_oof_error"].sum()
        )

        capture = (
            errors_captured / total_errors
            if total_errors else 0
        )

        print(
            f"Top {frac:>5.0%} uncertainty | "
            f"n={n:3d} | "
            f"errors={errors_captured:3d} | "
            f"error capture={capture:.3f}"
        )

print("\nPASS — R4 uncertainty separation audit complete")

R4-09 — UNCERTAINTY ERROR-SEPARATION AUDIT

OOF errors: 62
OOF correct: 778

Error-detection discrimination
----------------------------------------------------------------------------------------------------
           uncertainty_signal  error_detection_auc
          1 - top1 confidence             0.903454
           normalized entropy             0.902956
         1 - top1/top2 margin             0.902169
1 - feasible probability mass             0.765217

Highest-uncertainty error concentration
----------------------------------------------------------------------------------------------------

1 - top1 confidence
Top    5% uncertainty | n= 42 | errors= 23 | error capture=0.371
Top   10% uncertainty | n= 84 | errors= 35 | error capture=0.565
Top   15% uncertainty | n=126 | errors= 46 | error capture=0.742
Top   20% uncertainty | n=168 | errors= 52 | error capture=0.839

1 - top1/top2 margin
Top    5% uncertainty | n= 42 | errors= 21 | error capture=0.339
Top   10% uncertainty | n=

In [152]:
# ====================================================================================================
# R4-10 — DEVELOPMENT-ONLY SELECTIVE RISK–COVERAGE SWEEP
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("R4-10 — DEVELOPMENT-ONLY SELECTIVE RISK–COVERAGE SWEEP")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

df = pd.read_csv(
    R4_OUT / "R4_oof_uncertainty_development.csv"
)

assert len(df) == 840

# Higher score = more uncertain = stronger escalation candidate
signals = {
    "top1":
        1.0 - df["r4_top1_confidence"],

    "margin":
        1.0 - df["r4_margin"],

    "entropy":
        df["r4_normalized_entropy"],

    "feasible_mass":
        1.0 - df["r4_feasible_mass"],
}

budgets = [
    0.00,
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
    0.175,
    0.20,
    0.225,
    0.25,
]

total_errors = int(df["r4_oof_error"].sum())

rows = []

for signal_name, score in signals.items():

    order = np.argsort(-np.asarray(score))

    for budget in budgets:

        n_escalate = int(round(len(df) * budget))

        escalated = np.zeros(len(df), dtype=bool)

        if n_escalate > 0:
            escalated[order[:n_escalate]] = True

        autonomous = ~escalated

        n_auto = int(autonomous.sum())
        n_esc = int(escalated.sum())

        auto_errors = int(
            df.loc[autonomous, "r4_oof_error"].sum()
        )

        esc_errors = int(
            df.loc[escalated, "r4_oof_error"].sum()
        )

        auto_correct = n_auto - auto_errors

        autonomous_accuracy = (
            auto_correct / n_auto
            if n_auto else np.nan
        )

        selective_risk = (
            auto_errors / n_auto
            if n_auto else np.nan
        )

        error_capture = (
            esc_errors / total_errors
            if total_errors else 0.0
        )

        error_precision = (
            esc_errors / n_esc
            if n_esc else np.nan
        )

        rows.append({
            "signal": signal_name,
            "target_escalation_budget": budget,
            "escalated_n": n_esc,
            "autonomous_n": n_auto,
            "coverage": n_auto / len(df),
            "escalation_rate": n_esc / len(df),
            "autonomous_accuracy": autonomous_accuracy,
            "selective_risk": selective_risk,
            "errors_captured": esc_errors,
            "errors_remaining_autonomous": auto_errors,
            "error_capture_rate": error_capture,
            "escalation_error_precision": error_precision,
        })

result = pd.DataFrame(rows)

out_path = (
    R4_OUT /
    "R4_development_risk_coverage_sweep.csv"
)

result.to_csv(out_path, index=False)

print("\nDEVELOPMENT RISK–COVERAGE TABLE")
print("-" * 100)

display_cols = [
    "signal",
    "escalation_rate",
    "coverage",
    "autonomous_accuracy",
    "selective_risk",
    "errors_captured",
    "errors_remaining_autonomous",
    "error_capture_rate",
]

print(
    result[display_cols]
    .to_string(index=False)
)

print("\n" + "=" * 100)
print("BEST SIGNAL AT EACH ESCALATION BUDGET")
print("=" * 100)

best = (
    result.sort_values(
        [
            "target_escalation_budget",
            "autonomous_accuracy",
            "error_capture_rate",
        ],
        ascending=[True, False, False]
    )
    .groupby(
        "target_escalation_budget",
        as_index=False
    )
    .first()
)

print(
    best[
        [
            "target_escalation_budget",
            "signal",
            "coverage",
            "autonomous_accuracy",
            "errors_captured",
            "error_capture_rate",
        ]
    ].to_string(index=False)
)

best.to_csv(
    R4_OUT /
    "R4_best_signal_by_budget.csv",
    index=False
)

print("\nPASS — development-only risk–coverage sweep complete")

R4-10 — DEVELOPMENT-ONLY SELECTIVE RISK–COVERAGE SWEEP

DEVELOPMENT RISK–COVERAGE TABLE
----------------------------------------------------------------------------------------------------
       signal  escalation_rate  coverage  autonomous_accuracy  selective_risk  errors_captured  errors_remaining_autonomous  error_capture_rate
         top1            0.000     1.000             0.926190        0.073810                0                           62            0.000000
         top1            0.025     0.975             0.936508        0.063492               10                           52            0.161290
         top1            0.050     0.950             0.951128        0.048872               23                           39            0.370968
         top1            0.075     0.925             0.954955        0.045045               27                           35            0.435484
         top1            0.100     0.900             0.964286        0.035714              

In [153]:
# ====================================================================================================
# R4-11 — R3 + UNCERTAINTY COMBINED DEVELOPMENT SWEEP
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json

print("=" * 100)
print("R4-11 — R3 + UNCERTAINTY COMBINED DEVELOPMENT SWEEP")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

R4_OUT = Path("/content/R4_work/R4_results")

df = pd.read_csv(
    R4_OUT / "R4_oof_uncertainty_development.csv"
)

policy_path = (
    BASE /
    "02_PROVENANCE" /
    "R3_frozen_oof_mass_policy.json"
)

with open(policy_path, "r") as f:
    r3_policy = json.load(f)

print("\nFrozen R3 OOF policy")
print("-" * 100)
print(json.dumps(r3_policy, indent=2))

# Locate frozen threshold safely
tau_candidates = []

def collect_tau(obj):
    if isinstance(obj, dict):
        for k, v in obj.items():
            if "tau" in str(k).lower() or "threshold" in str(k).lower():
                if isinstance(v, (int, float)):
                    tau_candidates.append((k, float(v)))
            collect_tau(v)
    elif isinstance(obj, list):
        for v in obj:
            collect_tau(v)

collect_tau(r3_policy)

print("\nNumeric threshold candidates:")
for item in tau_candidates:
    print(item)

# We know the frozen R3 tau from the verified package.
R3_TAU = 0.5486315987262682

print("\nUsing verified frozen R3 tau:", R3_TAU)

# R3 escalation:
# empty candidate set OR feasible mass below frozen tau.
r3_escalate = (
    (df["candidate_count_oof"] == 0) |
    (df["r4_feasible_mass"] < R3_TAU)
)

print("\nR3 OOF baseline")
print("-" * 100)

print("Total rows       :", len(df))
print("R3 escalated     :", int(r3_escalate.sum()))
print("R3 escalation %  :", r3_escalate.mean())
print("R3 coverage      :", (~r3_escalate).mean())

r3_auto = ~r3_escalate

r3_auto_errors = int(
    df.loc[r3_auto, "r4_oof_error"].sum()
)

r3_auto_n = int(r3_auto.sum())

print(
    "R3 autonomous accuracy:",
    1.0 - r3_auto_errors / r3_auto_n
)

print(
    "R3 autonomous errors:",
    r3_auto_errors
)

# -------------------------------------------------------------------------
# Add uncertainty escalation ONLY among cases R3 would otherwise automate.
# -------------------------------------------------------------------------

signals = {
    "top1":
        1.0 - df["r4_top1_confidence"],

    "margin":
        1.0 - df["r4_margin"],

    "entropy":
        df["r4_normalized_entropy"],
}

# Additional escalation budget is expressed as fraction of ALL 840 cases.
additional_budgets = [
    0.025,
    0.05,
    0.075,
    0.10,
    0.125,
    0.15,
]

rows = []

for signal_name, score in signals.items():

    eligible_idx = np.where(~r3_escalate)[0]

    eligible_scores = np.asarray(score)[eligible_idx]

    order_local = np.argsort(-eligible_scores)

    ordered_idx = eligible_idx[order_local]

    for extra_budget in additional_budgets:

        requested_extra_n = int(
            round(len(df) * extra_budget)
        )

        extra_n = min(
            requested_extra_n,
            len(ordered_idx)
        )

        extra_escalate = np.zeros(
            len(df),
            dtype=bool
        )

        if extra_n:
            extra_escalate[
                ordered_idx[:extra_n]
            ] = True

        combined_escalate = (
            r3_escalate |
            extra_escalate
        )

        autonomous = ~combined_escalate

        n_auto = int(autonomous.sum())
        n_esc = int(combined_escalate.sum())

        auto_errors = int(
            df.loc[
                autonomous,
                "r4_oof_error"
            ].sum()
        )

        esc_errors = int(
            df.loc[
                combined_escalate,
                "r4_oof_error"
            ].sum()
        )

        extra_errors = int(
            df.loc[
                extra_escalate,
                "r4_oof_error"
            ].sum()
        )

        rows.append({
            "signal": signal_name,
            "additional_budget": extra_budget,
            "r3_escalated_n": int(r3_escalate.sum()),
            "uncertainty_added_n": int(extra_escalate.sum()),
            "combined_escalated_n": n_esc,
            "combined_escalation_rate": n_esc / len(df),
            "combined_coverage": n_auto / len(df),
            "autonomous_n": n_auto,
            "autonomous_errors": auto_errors,
            "autonomous_accuracy":
                1.0 - auto_errors / n_auto,
            "all_errors_captured": esc_errors,
            "all_error_capture_rate":
                esc_errors / int(df["r4_oof_error"].sum()),
            "additional_errors_captured":
                extra_errors,
            "additional_error_precision":
                extra_errors / extra_n if extra_n else np.nan,
        })

combined = pd.DataFrame(rows)

combined.to_csv(
    R4_OUT /
    "R4_r3_plus_uncertainty_development_sweep.csv",
    index=False
)

print("\nR3 + uncertainty results")
print("-" * 100)

print(
    combined[
        [
            "signal",
            "additional_budget",
            "combined_escalation_rate",
            "combined_coverage",
            "autonomous_accuracy",
            "autonomous_errors",
            "all_error_capture_rate",
            "additional_errors_captured",
        ]
    ].to_string(index=False)
)

print("\nPASS — R3 + uncertainty development sweep complete")

R4-11 — R3 + UNCERTAINTY COMBINED DEVELOPMENT SWEEP

Frozen R3 OOF policy
----------------------------------------------------------------------------------------------------
{
  "calibration_folds": 5,
  "calibration_method": "5-fold stratified out-of-fold calibration on the frozen 840-row training split",
  "calibration_nonempty_rows": 830,
  "calibration_seed": 42,
  "calibration_strict_empty_rows": 10,
  "calibration_total_rows": 840,
  "comparison_operator": "feasible_probability_mass < threshold",
  "experiment": "R3 Candidate Masking",
  "frozen_feasible_mass_threshold": 0.5486315987262682,
  "policy": {
    "empty_feasible_set": "ESCALATE_EMPTY",
    "nonempty_feasible_set_below_threshold": "ESCALATE_LOW_FEASIBLE_MASS",
    "otherwise": "STRICT_MASKED_ARGMAX"
  },
  "policy_name": "strict_v2_mask_plus_oof_feasible_mass_escalation",
  "status": "FROZEN_BEFORE_FINAL_HELDOUT_EVALUATION",
  "target_low_mass_quantile": 0.05,
  "threshold_selection_uses_test_labels": false,
  "thresh

In [154]:
# ====================================================================================================
# R4-12 — DERIVE FIXED R4 UNCERTAINTY THRESHOLD FROM OOF DEVELOPMENT DATA
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 100)
print("R4-12 — DERIVE FIXED R4 UNCERTAINTY THRESHOLD")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

df = pd.read_csv(
    R4_OUT / "R4_oof_uncertainty_development.csv"
)

# -------------------------------------------------------------------------
# Frozen R3 policy
# -------------------------------------------------------------------------

R3_TAU = 0.5486315987262682

r3_escalate = (
    (df["candidate_count_oof"] == 0) |
    (df["r4_feasible_mass"] < R3_TAU)
)

eligible = df.loc[~r3_escalate].copy()

print("\nR3 DEVELOPMENT STATE")
print("-" * 100)

print("Total development rows :", len(df))
print("R3 escalated           :", int(r3_escalate.sum()))
print("R3 autonomous          :", len(eligible))

# -------------------------------------------------------------------------
# Freeze R4 design choice
#
# Signal: top1-top2 margin
# Target additional escalation budget: 10% of ALL development rows
# -------------------------------------------------------------------------

TARGET_EXTRA_FRACTION = 0.10

target_n = int(round(len(df) * TARGET_EXTRA_FRACTION))

print("\nR4 DESIGN CHOICE")
print("-" * 100)

print("Signal                     : top1-top2 margin")
print("Target additional fraction :", TARGET_EXTRA_FRACTION)
print("Target additional rows     :", target_n)

# Lower margin = higher uncertainty.
ordered = eligible.sort_values(
    ["r4_margin", "train_id"],
    ascending=[True, True]
).reset_index(drop=True)

assert target_n > 0
assert target_n < len(ordered)

# Threshold exactly between the last escalated and first autonomous margin.
last_escalated_margin = float(
    ordered.loc[target_n - 1, "r4_margin"]
)

first_autonomous_margin = float(
    ordered.loc[target_n, "r4_margin"]
)

threshold = (
    last_escalated_margin +
    first_autonomous_margin
) / 2.0

print("\nTHRESHOLD DERIVATION")
print("-" * 100)

print("Last escalated margin :", last_escalated_margin)
print("First autonomous margin:", first_autonomous_margin)
print("Frozen margin threshold:", threshold)

assert last_escalated_margin <= threshold
assert threshold < first_autonomous_margin

# R4 rule:
# R3 escalation OR margin < frozen threshold
uncertainty_escalate = (
    (~r3_escalate) &
    (df["r4_margin"] < threshold)
)

combined_escalate = (
    r3_escalate |
    uncertainty_escalate
)

autonomous = ~combined_escalate

print("\nDEVELOPMENT REPRODUCTION")
print("-" * 100)

print("R3 escalations          :", int(r3_escalate.sum()))
print("Added uncertainty       :", int(uncertainty_escalate.sum()))
print("Total escalations       :", int(combined_escalate.sum()))
print("Coverage                :", autonomous.mean())
print("Escalation rate         :", combined_escalate.mean())

auto_errors = int(
    df.loc[autonomous, "r4_oof_error"].sum()
)

auto_n = int(autonomous.sum())

auto_accuracy = 1.0 - auto_errors / auto_n

print("Autonomous n            :", auto_n)
print("Autonomous errors       :", auto_errors)
print("Autonomous accuracy     :", auto_accuracy)

assert int(uncertainty_escalate.sum()) == target_n
assert auto_errors == 13

print("\nPASS — selected R4 development operating point reproduced")

# -------------------------------------------------------------------------
# Freeze policy BEFORE held-out evaluation
# -------------------------------------------------------------------------

policy = {
    "experiment": "R4 Uncertainty-Aware Selective Escalation and Robustness",
    "status": "FROZEN_BEFORE_R4_HELDOUT_EVALUATION",

    "base_policy": "R3 strict-v2 mask + frozen OOF feasible-mass escalation",

    "r3_frozen_feasible_mass_threshold": R3_TAU,

    "uncertainty_signal": "top1_minus_top2_probability_margin",

    "uncertainty_direction":
        "lower_margin_means_higher_uncertainty",

    "uncertainty_rule":
        "if R3 would automate and margin < frozen_margin_threshold, escalate",

    "target_additional_escalation_fraction_of_development": TARGET_EXTRA_FRACTION,

    "development_total_n": int(len(df)),
    "development_r3_autonomous_n": int((~r3_escalate).sum()),
    "development_added_uncertainty_n": int(uncertainty_escalate.sum()),

    "frozen_margin_threshold": threshold,

    "threshold_last_escalated_margin": last_escalated_margin,
    "threshold_first_autonomous_margin": first_autonomous_margin,

    "threshold_selection_uses_r4_heldout_labels": False,
    "threshold_selection_uses_r4_heldout_metrics": False,

    "development_combined_coverage": float(autonomous.mean()),
    "development_combined_escalation_rate": float(combined_escalate.mean()),
    "development_autonomous_accuracy": float(auto_accuracy),
    "development_autonomous_errors": auto_errors,
}

policy_path = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

with open(policy_path, "w") as f:
    json.dump(
        policy,
        f,
        indent=2,
        sort_keys=True
    )

policy_sha = hashlib.sha256(
    policy_path.read_bytes()
).hexdigest()

print("\nFROZEN R4 POLICY")
print("-" * 100)
print(json.dumps(policy, indent=2))

print("\nPolicy file :", policy_path)
print("SHA256      :", policy_sha)

print("\nPASS — R4 uncertainty policy frozen BEFORE held-out evaluation")

R4-12 — DERIVE FIXED R4 UNCERTAINTY THRESHOLD

R3 DEVELOPMENT STATE
----------------------------------------------------------------------------------------------------
Total development rows : 840
R3 escalated           : 52
R3 autonomous          : 788

R4 DESIGN CHOICE
----------------------------------------------------------------------------------------------------
Signal                     : top1-top2 margin
Target additional fraction : 0.1
Target additional rows     : 84

THRESHOLD DERIVATION
----------------------------------------------------------------------------------------------------
Last escalated margin : 0.7087780535254092
First autonomous margin: 0.7107493848751814
Frozen margin threshold: 0.7097637192002952

DEVELOPMENT REPRODUCTION
----------------------------------------------------------------------------------------------------
R3 escalations          : 52
Added uncertainty       : 84
Total escalations       : 136
Coverage                : 0.8380952380952381
E

In [155]:
# ====================================================================================================
# R4-13 — PRE-HELDOUT FREEZE CHECKPOINT
# ====================================================================================================

from pathlib import Path
import hashlib
import json

print("=" * 100)
print("R4-13 — PRE-HELDOUT FREEZE CHECKPOINT")
print("=" * 100)

policy_path = Path(
    "/content/R4_work/R4_results/"
    "R4_frozen_uncertainty_policy.json"
)

assert policy_path.exists()

raw = policy_path.read_bytes()
sha = hashlib.sha256(raw).hexdigest()

with open(policy_path, "r") as f:
    policy = json.load(f)

assert policy["status"] == "FROZEN_BEFORE_R4_HELDOUT_EVALUATION"
assert policy["threshold_selection_uses_r4_heldout_labels"] is False
assert policy["threshold_selection_uses_r4_heldout_metrics"] is False

print("\nPolicy SHA256:")
print(sha)

print("\nFrozen margin threshold:")
print(policy["frozen_margin_threshold"])

print("\nBase R3 tau:")
print(policy["r3_frozen_feasible_mass_threshold"])

print("\nDevelopment coverage:")
print(policy["development_combined_coverage"])

print("\nDevelopment autonomous accuracy:")
print(policy["development_autonomous_accuracy"])

checkpoint = {
    "policy_sha256": sha,
    "policy_status": policy["status"],
    "heldout_evaluated": False
}

checkpoint_path = Path(
    "/content/R4_work/R4_results/"
    "R4_PRE_HELDOUT_FREEZE_CHECKPOINT.json"
)

with open(checkpoint_path, "w") as f:
    json.dump(
        checkpoint,
        f,
        indent=2,
        sort_keys=True
    )

print("\nCheckpoint:", checkpoint_path)

print("\n" + "=" * 100)
print("PASS — R4 POLICY IS NOW FROZEN")
print("HELD-OUT EVALUATION MAY BEGIN ONLY AFTER THIS CHECKPOINT")
print("=" * 100)

R4-13 — PRE-HELDOUT FREEZE CHECKPOINT

Policy SHA256:
1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42

Frozen margin threshold:
0.7097637192002952

Base R3 tau:
0.5486315987262682

Development coverage:
0.8380952380952381

Development autonomous accuracy:
0.9815340909090909

Checkpoint: /content/R4_work/R4_results/R4_PRE_HELDOUT_FREEZE_CHECKPOINT.json

PASS — R4 POLICY IS NOW FROZEN
HELD-OUT EVALUATION MAY BEGIN ONLY AFTER THIS CHECKPOINT


In [156]:
# ====================================================================================================
# R4-15 — R4 HELD-OUT ERROR-CAPTURE AUDIT
# ====================================================================================================

from pathlib import Path
import pandas as pd

print("=" * 100)
print("R4-15 — HELD-OUT ERROR-CAPTURE AUDIT")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

df = pd.read_csv(
    R4_OUT /
    "R4_frozen_policy_heldout_decisions.csv"
)

# Cases R3 automated but got wrong
r3_remaining_errors = df[
    (~df["r4_r3_escalated"]) &
    (~df["r4_final_correct"])
].copy()

print("\nR3 REMAINING ERRORS ENTERING R4")
print("-" * 100)

print("Count:", len(r3_remaining_errors))

cols = [
    "id",
    "truth",
    "final_pred",
    "uc_kind",
    "candidate_count",
    "feasible_probability_mass",
    "r4_top1_confidence",
    "r4_margin",
    "r4_normalized_entropy",
    "r4_uncertainty_escalated",
    "r4_escalation_reason",
]

print(
    r3_remaining_errors[cols]
    .sort_values("r4_margin")
    .to_string(index=False)
)

print("\n" + "=" * 100)
print("ERROR CAPTURE BY UC KIND")
print("=" * 100)

audit = (
    r3_remaining_errors
    .groupby("uc_kind")
    .agg(
        r3_remaining_errors=("id", "count"),
        captured_by_r4=(
            "r4_uncertainty_escalated",
            "sum"
        )
    )
    .reset_index()
)

audit["capture_rate"] = (
    audit["captured_by_r4"] /
    audit["r3_remaining_errors"]
)

print(audit.to_string(index=False))

audit.to_csv(
    R4_OUT /
    "R4_heldout_error_capture_by_uc_kind.csv",
    index=False
)

print("\n" + "=" * 100)
print("UNCERTAINTY ESCALATIONS: CORRECT VS ERROR")
print("=" * 100)

esc = df[df["r4_uncertainty_escalated"]].copy()

esc_summary = (
    esc.groupby("r4_final_correct")
    .size()
    .rename("n")
    .reset_index()
)

print(esc_summary.to_string(index=False))

print("\nPASS — R4 held-out error-capture audit complete")

R4-15 — HELD-OUT ERROR-CAPTURE AUDIT


FileNotFoundError: [Errno 2] No such file or directory: '/content/R4_work/R4_results/R4_frozen_policy_heldout_decisions.csv'

In [157]:
from pathlib import Path

print("=" * 100)
print("R4-14A — CHECK WHETHER R4-14 OUTPUT EXISTS")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

files = [
    "R4_frozen_uncertainty_policy.json",
    "R4_PRE_HELDOUT_FREEZE_CHECKPOINT.json",
    "R4_frozen_policy_heldout_decisions.csv",
    "R4_frozen_policy_heldout_summary.json",
]

for name in files:
    p = R4_OUT / name
    print(
        ("PASS   " if p.exists() else "MISSING"),
        name
    )

R4-14A — CHECK WHETHER R4-14 OUTPUT EXISTS
PASS    R4_frozen_uncertainty_policy.json
PASS    R4_PRE_HELDOUT_FREEZE_CHECKPOINT.json
MISSING R4_frozen_policy_heldout_decisions.csv
MISSING R4_frozen_policy_heldout_summary.json


In [158]:
# ====================================================================================================
# R4-14 — ONE-SHOT FROZEN R4 HELD-OUT EVALUATION
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np
import json
import hashlib

print("=" * 100)
print("R4-14 — ONE-SHOT FROZEN R4 HELD-OUT EVALUATION")
print("=" * 100)

BASE = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

R4_OUT = Path("/content/R4_work/R4_results")

POLICY = R4_OUT / "R4_frozen_uncertainty_policy.json"
CHECKPOINT = R4_OUT / "R4_PRE_HELDOUT_FREEZE_CHECKPOINT.json"

HELDOUT = (
    BASE /
    "03_RESULTS_RAW" /
    "R3_final_frozen_policy_heldout_decisions.csv"
)

# ----------------------------------------------------------------------------------------------------
# 1. Verify all prerequisites
# ----------------------------------------------------------------------------------------------------

assert POLICY.exists(), "STOP: frozen R4 policy missing"
assert CHECKPOINT.exists(), "STOP: pre-heldout checkpoint missing"
assert HELDOUT.exists(), "STOP: frozen R3 held-out evidence missing"

EXPECTED_POLICY_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

policy_sha_before = hashlib.sha256(
    POLICY.read_bytes()
).hexdigest()

assert policy_sha_before == EXPECTED_POLICY_SHA, (
    "STOP: frozen R4 policy SHA changed"
)

with open(POLICY, "r") as f:
    policy = json.load(f)

with open(CHECKPOINT, "r") as f:
    checkpoint = json.load(f)

assert checkpoint["policy_sha256"] == EXPECTED_POLICY_SHA
assert checkpoint["heldout_evaluated"] is False

R3_TAU = float(
    policy["r3_frozen_feasible_mass_threshold"]
)

R4_MARGIN_TAU = float(
    policy["frozen_margin_threshold"]
)

print("\nPRE-EVALUATION FREEZE VERIFICATION")
print("-" * 100)
print("Policy SHA256 :", policy_sha_before)
print("R3 tau        :", R3_TAU)
print("R4 margin tau :", R4_MARGIN_TAU)

# ----------------------------------------------------------------------------------------------------
# 2. Load frozen held-out evidence
# ----------------------------------------------------------------------------------------------------

df = pd.read_csv(HELDOUT)

assert len(df) == 360
assert df["id"].nunique() == 360

required = [
    "truth",
    "p_eMBB",
    "p_URLLC",
    "p_mMTC",
    "final_action",
    "final_pred",
    "final_escalation_reason",
    "candidate_count",
    "feasible_probability_mass",
    "uc_kind",
]

missing = [c for c in required if c not in df.columns]

assert not missing, f"STOP: missing columns: {missing}"

print("\nHELD-OUT EVIDENCE")
print("-" * 100)
print("Rows       :", len(df))
print("Unique IDs :", df["id"].nunique())

# ----------------------------------------------------------------------------------------------------
# 3. Compute uncertainty WITHOUT using truth labels
# ----------------------------------------------------------------------------------------------------

P = df[
    ["p_eMBB", "p_URLLC", "p_mMTC"]
].to_numpy(dtype=float)

assert np.all(P >= 0.0)
assert np.all(P <= 1.0)

row_sums = P.sum(axis=1)

assert np.allclose(
    row_sums,
    1.0,
    atol=1e-6
)

sorted_p = np.sort(P, axis=1)

df["r4_top1_confidence"] = sorted_p[:, -1]
df["r4_top2_confidence"] = sorted_p[:, -2]

df["r4_margin"] = (
    df["r4_top1_confidence"] -
    df["r4_top2_confidence"]
)

entropy = -np.sum(
    P * np.log(
        np.clip(P, 1e-15, 1.0)
    ),
    axis=1
)

df["r4_normalized_entropy"] = (
    entropy / np.log(3.0)
)

# ----------------------------------------------------------------------------------------------------
# 4. Preserve exact frozen R3 decision state
# ----------------------------------------------------------------------------------------------------

r3_escalated = (
    df["final_action"]
    .astype(str)
    .str.strip()
    .str.upper()
    .eq("ESCALATE")
)

r3_autonomous = ~r3_escalated

assert int(r3_escalated.sum()) == 19, (
    f"STOP: expected 19 R3 escalations, got {int(r3_escalated.sum())}"
)

assert int(r3_autonomous.sum()) == 341

# R3 correctness exists only for autonomous predictions.
r3_correct = (
    r3_autonomous &
    df["final_pred"].notna() &
    (
        df["final_pred"].astype(str) ==
        df["truth"].astype(str)
    )
)

r3_wrong = (
    r3_autonomous &
    (~r3_correct)
)

r3_correct_n = int(r3_correct.sum())
r3_error_n = int(r3_wrong.sum())

assert r3_correct_n == 329, (
    f"STOP: expected 329 correct R3 autonomous decisions, got {r3_correct_n}"
)

assert r3_error_n == 12, (
    f"STOP: expected 12 R3 autonomous errors, got {r3_error_n}"
)

print("\nFROZEN R3 HELD-OUT BASELINE")
print("-" * 100)
print("Escalated             :", int(r3_escalated.sum()))
print("Autonomous            :", int(r3_autonomous.sum()))
print("Autonomous correct    :", r3_correct_n)
print("Autonomous errors     :", r3_error_n)
print("Autonomous accuracy   :", r3_correct_n / int(r3_autonomous.sum()))
print("Coverage              :", r3_autonomous.mean())

# ----------------------------------------------------------------------------------------------------
# 5. Apply EXACT frozen R4 uncertainty policy
#
# R4 rule:
#   - preserve every R3 escalation
#   - among cases R3 would automate:
#       escalate if top1-top2 margin < 0.7097637192002952
# ----------------------------------------------------------------------------------------------------

r4_uncertainty_escalated = (
    r3_autonomous &
    (df["r4_margin"] < R4_MARGIN_TAU)
)

r4_escalated = (
    r3_escalated |
    r4_uncertainty_escalated
)

r4_autonomous = ~r4_escalated

# ----------------------------------------------------------------------------------------------------
# 6. Evaluate frozen R4 policy
# ----------------------------------------------------------------------------------------------------

r4_correct = (
    r4_autonomous &
    df["final_pred"].notna() &
    (
        df["final_pred"].astype(str) ==
        df["truth"].astype(str)
    )
)

r4_wrong = (
    r4_autonomous &
    (~r4_correct)
)

added_n = int(r4_uncertainty_escalated.sum())

captured_errors = int(
    (
        r4_uncertainty_escalated &
        r3_wrong
    ).sum()
)

correct_cases_escalated = int(
    (
        r4_uncertainty_escalated &
        r3_correct
    ).sum()
)

r4_auto_n = int(r4_autonomous.sum())
r4_correct_n = int(r4_correct.sum())
r4_error_n = int(r4_wrong.sum())

r4_total_escalated = int(r4_escalated.sum())

coverage = r4_auto_n / len(df)

escalation_rate = (
    r4_total_escalated / len(df)
)

autonomous_accuracy = (
    r4_correct_n / r4_auto_n
)

selective_risk = (
    r4_error_n / r4_auto_n
)

residual_error_capture_rate = (
    captured_errors / r3_error_n
)

added_escalation_precision = (
    captured_errors / added_n
    if added_n else 0.0
)

print("\n" + "=" * 100)
print("FROZEN R4 HELD-OUT RESULT")
print("=" * 100)

print("Added uncertainty escalations :", added_n)
print("Errors captured               :", captured_errors)
print("Correct cases escalated       :", correct_cases_escalated)
print()
print("Total escalated               :", r4_total_escalated)
print("Autonomous                    :", r4_auto_n)
print("Coverage                      :", coverage)
print("Escalation rate               :", escalation_rate)
print()
print("Autonomous correct            :", r4_correct_n)
print("Autonomous errors             :", r4_error_n)
print("Autonomous accuracy           :", autonomous_accuracy)
print("Selective risk                :", selective_risk)
print()
print("R3 residual error capture     :", residual_error_capture_rate)
print("Added escalation precision    :", added_escalation_precision)

# ----------------------------------------------------------------------------------------------------
# 7. Build immutable row-level R4 evidence
# ----------------------------------------------------------------------------------------------------

df["r4_r3_escalated"] = r3_escalated
df["r4_r3_autonomous"] = r3_autonomous
df["r4_r3_correct"] = r3_correct
df["r4_r3_wrong"] = r3_wrong

df["r4_uncertainty_escalated"] = (
    r4_uncertainty_escalated
)

df["r4_final_escalated"] = r4_escalated
df["r4_final_autonomous"] = r4_autonomous
df["r4_final_correct"] = r4_correct
df["r4_final_wrong"] = r4_wrong

df["r4_escalation_reason"] = np.select(
    [
        r3_escalated,
        r4_uncertainty_escalated,
    ],
    [
        "R3_FROZEN_ESCALATION",
        "LOW_MARGIN_UNCERTAINTY",
    ],
    default="AUTONOMOUS"
)

ROW_OUT = (
    R4_OUT /
    "R4_frozen_policy_heldout_decisions.csv"
)

df.to_csv(
    ROW_OUT,
    index=False
)

# ----------------------------------------------------------------------------------------------------
# 8. Save frozen held-out summary
# ----------------------------------------------------------------------------------------------------

summary = {
    "experiment":
        "R4 Uncertainty-Aware Selective Escalation and Robustness",

    "evaluation_status":
        "ONE_SHOT_FROZEN_HELDOUT_EVALUATION",

    "n_total": int(len(df)),

    "r3_escalated_n":
        int(r3_escalated.sum()),

    "r3_autonomous_n":
        int(r3_autonomous.sum()),

    "r3_autonomous_correct_n":
        r3_correct_n,

    "r3_autonomous_errors":
        r3_error_n,

    "r3_coverage":
        float(r3_autonomous.mean()),

    "r3_autonomous_accuracy":
        float(
            r3_correct_n /
            int(r3_autonomous.sum())
        ),

    "r4_added_uncertainty_escalations":
        added_n,

    "r4_added_errors_captured":
        captured_errors,

    "r4_added_correct_cases_escalated":
        correct_cases_escalated,

    "r4_total_escalated_n":
        r4_total_escalated,

    "r4_autonomous_n":
        r4_auto_n,

    "r4_coverage":
        float(coverage),

    "r4_escalation_rate":
        float(escalation_rate),

    "r4_autonomous_correct_n":
        r4_correct_n,

    "r4_autonomous_errors":
        r4_error_n,

    "r4_autonomous_accuracy":
        float(autonomous_accuracy),

    "r4_selective_risk":
        float(selective_risk),

    "r4_error_capture_rate_from_r3_residual":
        float(residual_error_capture_rate),

    "r4_added_escalation_error_precision":
        float(added_escalation_precision),

    "frozen_r3_tau":
        R3_TAU,

    "frozen_r4_margin_threshold":
        R4_MARGIN_TAU,

    "policy_sha256":
        policy_sha_before,

    "threshold_changed_after_heldout":
        False,

    "signal_changed_after_heldout":
        False,
}

SUMMARY_OUT = (
    R4_OUT /
    "R4_frozen_policy_heldout_summary.json"
)

with open(SUMMARY_OUT, "w") as f:
    json.dump(
        summary,
        f,
        indent=2,
        sort_keys=True
    )

# ----------------------------------------------------------------------------------------------------
# 9. Hash evidence
# ----------------------------------------------------------------------------------------------------

row_sha = hashlib.sha256(
    ROW_OUT.read_bytes()
).hexdigest()

summary_sha = hashlib.sha256(
    SUMMARY_OUT.read_bytes()
).hexdigest()

policy_sha_after = hashlib.sha256(
    POLICY.read_bytes()
).hexdigest()

assert policy_sha_after == policy_sha_before
assert policy_sha_after == EXPECTED_POLICY_SHA

print("\n" + "=" * 100)
print("FROZEN EVIDENCE")
print("=" * 100)

print("Policy SHA256:")
print(policy_sha_after)

print("\nHeld-out decisions SHA256:")
print(row_sha)

print("\nHeld-out summary SHA256:")
print(summary_sha)

print("\nFiles:")
print(ROW_OUT)
print(SUMMARY_OUT)

print("\n" + "=" * 100)
print("PASS — R4 ONE-SHOT HELD-OUT EVALUATION COMPLETE")
print("PASS — FROZEN POLICY REMAINED UNCHANGED")
print("=" * 100)

R4-14 — ONE-SHOT FROZEN R4 HELD-OUT EVALUATION

PRE-EVALUATION FREEZE VERIFICATION
----------------------------------------------------------------------------------------------------
Policy SHA256 : 1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42
R3 tau        : 0.5486315987262682
R4 margin tau : 0.7097637192002952

HELD-OUT EVIDENCE
----------------------------------------------------------------------------------------------------
Rows       : 360
Unique IDs : 360

FROZEN R3 HELD-OUT BASELINE
----------------------------------------------------------------------------------------------------
Escalated             : 19
Autonomous            : 341
Autonomous correct    : 329
Autonomous errors     : 12
Autonomous accuracy   : 0.9648093841642229
Coverage              : 0.9472222222222222

FROZEN R4 HELD-OUT RESULT
Added uncertainty escalations : 35
Errors captured               : 7
Correct cases escalated       : 28

Total escalated               : 54
Autonomous        

In [159]:
# ====================================================================================================
# R4-15 — HELD-OUT ERROR-CAPTURE AND FAILURE-MODE AUDIT
# ====================================================================================================

from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 100)
print("R4-15 — HELD-OUT ERROR-CAPTURE AND FAILURE-MODE AUDIT")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

src = (
    R4_OUT /
    "R4_frozen_policy_heldout_decisions.csv"
)

assert src.exists()

df = pd.read_csv(src)

assert len(df) == 360

# ----------------------------------------------------------------------------------------------------
# 1. Recover the 12 errors that survived R3
# ----------------------------------------------------------------------------------------------------

r3_errors = df[
    df["r4_r3_wrong"].astype(bool)
].copy()

assert len(r3_errors) == 12

print("\nR3 RESIDUAL ERRORS ENTERING R4")
print("-" * 100)
print("Count:", len(r3_errors))

cols = [
    "id",
    "truth",
    "final_pred",
    "uc_kind",
    "candidate_count",
    "feasible_probability_mass",
    "r4_top1_confidence",
    "r4_margin",
    "r4_normalized_entropy",
    "r4_uncertainty_escalated",
]

print(
    r3_errors[cols]
    .sort_values(
        "r4_margin",
        ascending=True
    )
    .to_string(index=False)
)

# ----------------------------------------------------------------------------------------------------
# 2. Captured vs missed R3 errors
# ----------------------------------------------------------------------------------------------------

captured = r3_errors[
    r3_errors["r4_uncertainty_escalated"]
    .astype(bool)
].copy()

missed = r3_errors[
    ~r3_errors["r4_uncertainty_escalated"]
    .astype(bool)
].copy()

assert len(captured) == 7
assert len(missed) == 5

print("\n" + "=" * 100)
print("R4 CAPTURED R3 ERRORS")
print("=" * 100)

print(
    captured[cols]
    .sort_values("r4_margin")
    .to_string(index=False)
)

print("\n" + "=" * 100)
print("R4 MISSED R3 ERRORS")
print("=" * 100)

print(
    missed[cols]
    .sort_values("r4_margin")
    .to_string(index=False)
)

# ----------------------------------------------------------------------------------------------------
# 3. Error capture by UC kind
# ----------------------------------------------------------------------------------------------------

uc_audit = (
    r3_errors
    .groupby("uc_kind", dropna=False)
    .agg(
        r3_errors=("id", "count"),
        captured_by_r4=(
            "r4_uncertainty_escalated",
            "sum"
        )
    )
    .reset_index()
)

uc_audit["missed_by_r4"] = (
    uc_audit["r3_errors"] -
    uc_audit["captured_by_r4"]
)

uc_audit["capture_rate"] = (
    uc_audit["captured_by_r4"] /
    uc_audit["r3_errors"]
)

print("\n" + "=" * 100)
print("ERROR CAPTURE BY UC KIND")
print("=" * 100)

print(uc_audit.to_string(index=False))

uc_audit.to_csv(
    R4_OUT /
    "R4_heldout_error_capture_by_uc_kind.csv",
    index=False
)

# ----------------------------------------------------------------------------------------------------
# 4. Added escalation audit
# ----------------------------------------------------------------------------------------------------

added = df[
    df["r4_uncertainty_escalated"]
    .astype(bool)
].copy()

assert len(added) == 35

added["was_r3_error"] = (
    added["r4_r3_wrong"].astype(bool)
)

added_audit = (
    added.groupby(
        ["uc_kind", "was_r3_error"],
        dropna=False
    )
    .size()
    .rename("n")
    .reset_index()
)

print("\n" + "=" * 100)
print("ADDED R4 ESCALATIONS BY UC KIND / CORRECTNESS")
print("=" * 100)

print(added_audit.to_string(index=False))

added_audit.to_csv(
    R4_OUT /
    "R4_added_escalation_audit.csv",
    index=False
)

# ----------------------------------------------------------------------------------------------------
# 5. Compare uncertainty statistics
# ----------------------------------------------------------------------------------------------------

groups = []

for name, part in [
    ("R3_error_captured", captured),
    ("R3_error_missed", missed),
    (
        "R3_correct_but_R4_escalated",
        added[
            ~added["was_r3_error"]
        ]
    ),
    (
        "R4_final_autonomous_correct",
        df[
            df["r4_final_autonomous"].astype(bool) &
            df["r4_final_correct"].astype(bool)
        ]
    ),
]:

    groups.append({
        "group": name,
        "n": len(part),
        "mean_top1_confidence":
            part["r4_top1_confidence"].mean(),
        "mean_margin":
            part["r4_margin"].mean(),
        "mean_normalized_entropy":
            part["r4_normalized_entropy"].mean(),
        "mean_feasible_mass":
            part["feasible_probability_mass"].mean(),
    })

group_summary = pd.DataFrame(groups)

print("\n" + "=" * 100)
print("UNCERTAINTY PROFILE BY OUTCOME")
print("=" * 100)

print(group_summary.to_string(index=False))

group_summary.to_csv(
    R4_OUT /
    "R4_uncertainty_profile_by_outcome.csv",
    index=False
)

# ----------------------------------------------------------------------------------------------------
# 6. Save captured/missed cases
# ----------------------------------------------------------------------------------------------------

captured.to_csv(
    R4_OUT /
    "R4_captured_r3_errors.csv",
    index=False
)

missed.to_csv(
    R4_OUT /
    "R4_missed_r3_errors.csv",
    index=False
)

print("\nFiles written:")
print("R4_heldout_error_capture_by_uc_kind.csv")
print("R4_added_escalation_audit.csv")
print("R4_uncertainty_profile_by_outcome.csv")
print("R4_captured_r3_errors.csv")
print("R4_missed_r3_errors.csv")

print("\n" + "=" * 100)
print("PASS — R4 HELD-OUT ERROR-CAPTURE AUDIT COMPLETE")
print("=" * 100)

R4-15 — HELD-OUT ERROR-CAPTURE AND FAILURE-MODE AUDIT

R3 RESIDUAL ERRORS ENTERING R4
----------------------------------------------------------------------------------------------------
Count: 12
  id truth final_pred uc_kind  candidate_count  feasible_probability_mass  r4_top1_confidence  r4_margin  r4_normalized_entropy  r4_uncertainty_escalated
1055  mMTC      URLLC generic                3                   1.000000            0.553921   0.165994               0.782793                      True
 645  eMBB      URLLC generic                2                   0.959297            0.573465   0.187633               0.743334                      True
  38  mMTC      URLLC  unseen                2                   0.806811            0.513018   0.219225               0.928345                      True
 748 URLLC       mMTC generic                3                   1.000000            0.619142   0.282321               0.728987                      True
 103 URLLC       eMBB  unseen    

In [160]:
# ====================================================================================================
# R4-16 — FROZEN R3 VS FROZEN R4 SYSTEM COMPARISON
# ====================================================================================================

from pathlib import Path
import pandas as pd

print("=" * 100)
print("R4-16 — FROZEN R3 VS FROZEN R4 SYSTEM COMPARISON")
print("=" * 100)

R4_OUT = Path("/content/R4_work/R4_results")

df = pd.read_csv(
    R4_OUT /
    "R4_frozen_policy_heldout_decisions.csv"
)

n = len(df)

r3_auto = df["r4_r3_autonomous"].astype(bool)
r4_auto = df["r4_final_autonomous"].astype(bool)

r3_correct = df["r4_r3_correct"].astype(bool)
r4_correct = df["r4_final_correct"].astype(bool)

comparison = pd.DataFrame([
    {
        "system": "Frozen R3",
        "total_n": n,
        "autonomous_n": int(r3_auto.sum()),
        "escalated_n": int((~r3_auto).sum()),
        "coverage": r3_auto.mean(),
        "escalation_rate": (~r3_auto).mean(),
        "autonomous_correct": int(r3_correct.sum()),
        "autonomous_errors":
            int((r3_auto & ~r3_correct).sum()),
        "autonomous_accuracy":
            r3_correct.sum() / r3_auto.sum(),
        "selective_risk":
            (r3_auto & ~r3_correct).sum() /
            r3_auto.sum(),
        "correct_autonomous_over_total":
            r3_correct.sum() / n,
    },
    {
        "system": "Frozen R4",
        "total_n": n,
        "autonomous_n": int(r4_auto.sum()),
        "escalated_n": int((~r4_auto).sum()),
        "coverage": r4_auto.mean(),
        "escalation_rate": (~r4_auto).mean(),
        "autonomous_correct": int(r4_correct.sum()),
        "autonomous_errors":
            int((r4_auto & ~r4_correct).sum()),
        "autonomous_accuracy":
            r4_correct.sum() / r4_auto.sum(),
        "selective_risk":
            (r4_auto & ~r4_correct).sum() /
            r4_auto.sum(),
        "correct_autonomous_over_total":
            r4_correct.sum() / n,
    }
])

comparison.to_csv(
    R4_OUT /
    "R4_final_system_comparison.csv",
    index=False
)

print("\nSYSTEM COMPARISON")
print("-" * 100)

print(comparison.to_string(index=False))

r3 = comparison.iloc[0]
r4 = comparison.iloc[1]

print("\n" + "=" * 100)
print("R4 EFFECT RELATIVE TO R3")
print("=" * 100)

print(
    "Coverage change                 :",
    r4["coverage"] - r3["coverage"]
)

print(
    "Autonomous accuracy change      :",
    r4["autonomous_accuracy"] -
    r3["autonomous_accuracy"]
)

print(
    "Selective risk change           :",
    r4["selective_risk"] -
    r3["selective_risk"]
)

print(
    "Residual errors removed         :",
    int(
        r3["autonomous_errors"] -
        r4["autonomous_errors"]
    )
)

risk_reduction = (
    1 -
    r4["selective_risk"] /
    r3["selective_risk"]
)

print(
    "Relative selective-risk reduction:",
    risk_reduction
)

print(
    "Correct-autonomous/total change :",
    r4["correct_autonomous_over_total"] -
    r3["correct_autonomous_over_total"]
)

print("\nPASS — frozen R3/R4 comparison complete")

R4-16 — FROZEN R3 VS FROZEN R4 SYSTEM COMPARISON

SYSTEM COMPARISON
----------------------------------------------------------------------------------------------------
   system  total_n  autonomous_n  escalated_n  coverage  escalation_rate  autonomous_correct  autonomous_errors  autonomous_accuracy  selective_risk  correct_autonomous_over_total
Frozen R3      360           341           19  0.947222         0.052778                 329                 12             0.964809        0.035191                       0.913889
Frozen R4      360           306           54  0.850000         0.150000                 301                  5             0.983660        0.016340                       0.836111

R4 EFFECT RELATIVE TO R3
Coverage change                 : -0.09722222222222221
Autonomous accuracy change      : 0.01885074655473129
Selective risk change           : -0.018850746554731377
Residual errors removed         : 7
Relative selective-risk reduction: 0.5356753812636166
Correct-au

In [161]:
# ====================================================================================================
# R4-17 — ROBUSTNESS ASSET AND DATASET AUDIT
# ====================================================================================================

from pathlib import Path
import pandas as pd
import os

print("=" * 110)
print("R4-17 — ROBUSTNESS ASSET AND DATASET AUDIT")
print("=" * 110)

ROOT = Path("/content/R4_work/recovered_R3")
R4_OUT = Path("/content/R4_work/R4_results")

# ----------------------------------------------------------------------------------------------------
# 1. Locate potentially useful datasets
# ----------------------------------------------------------------------------------------------------

print("\nDATASET / TABLE FILES")
print("-" * 110)

table_files = []

for ext in ["*.csv", "*.json", "*.jsonl", "*.parquet"]:
    table_files.extend(ROOT.rglob(ext))

for p in sorted(set(table_files)):
    try:
        size = p.stat().st_size
    except:
        size = -1

    print(
        f"{str(p.relative_to(ROOT)):85s} "
        f"{size:>12,} bytes"
    )

# ----------------------------------------------------------------------------------------------------
# 2. Locate source/model/notebook assets
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SOURCE / MODEL / NOTEBOOK ASSETS")
print("=" * 110)

source_exts = [
    "*.py",
    "*.ipynb",
    "*.pkl",
    "*.joblib",
    "*.pickle",
    "*.pt",
    "*.pth",
    "*.onnx",
]

source_files = []

for ext in source_exts:
    source_files.extend(ROOT.rglob(ext))

if source_files:

    for p in sorted(set(source_files)):
        print(
            f"{str(p.relative_to(ROOT)):85s} "
            f"{p.stat().st_size:>12,} bytes"
        )

else:
    print("NONE FOUND")

# ----------------------------------------------------------------------------------------------------
# 3. Find frozen split files
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FROZEN SPLIT CANDIDATES")
print("=" * 110)

split_candidates = []

for p in ROOT.rglob("*.csv"):

    name = p.name.lower()

    if any(
        token in name
        for token in [
            "split",
            "test",
            "train",
            "heldout",
            "frozen1200"
        ]
    ):
        split_candidates.append(p)

for p in sorted(split_candidates):

    try:
        x = pd.read_csv(p)

        print("\nFILE:")
        print(p)

        print("Rows   :", len(x))
        print("Columns:", len(x.columns))

        print("COLUMN NAMES:")
        print(list(x.columns))

        print("\nFIRST ROW:")
        print(x.head(1).to_string(index=False))

    except Exception as e:

        print("\nFILE:")
        print(p)

        print("READ ERROR:", repr(e))

# ----------------------------------------------------------------------------------------------------
# 4. Specifically search for expected R3 frozen test split
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SEARCH FOR R3_frozen1200_test_split.csv")
print("=" * 110)

matches = list(
    ROOT.rglob("R3_frozen1200_test_split.csv")
)

if not matches:
    print("NOT FOUND")

else:

    for p in matches:

        print("\nFOUND:")
        print(p)

        test = pd.read_csv(p)

        print("\nShape:")
        print(test.shape)

        print("\nColumns:")
        for i, c in enumerate(test.columns):
            print(f"{i:3d}  {c}")

        print("\nDtypes:")
        print(test.dtypes.to_string())

        print("\nMissing values:")
        print(
            test.isna()
            .sum()
            .sort_values(ascending=False)
            .head(20)
            .to_string()
        )

        print("\nFirst 3 rows:")
        print(test.head(3).to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# 5. Look for likely text / KPI / intent fields in all candidate datasets
# ----------------------------------------------------------------------------------------------------

keywords = [
    "text",
    "intent",
    "query",
    "utterance",
    "prompt",
    "description",
    "latency",
    "throughput",
    "reliability",
    "bandwidth",
    "mobility",
    "device",
    "density",
    "kpi",
    "service",
    "slice",
]

print("\n" + "=" * 110)
print("LIKELY ROBUSTNESS-RELEVANT COLUMNS")
print("=" * 110)

for p in sorted(split_candidates):

    try:
        x = pd.read_csv(p, nrows=5)
    except:
        continue

    relevant = [
        c for c in x.columns
        if any(k in c.lower() for k in keywords)
    ]

    if relevant:
        print("\n", p.name)
        print(relevant)

# ----------------------------------------------------------------------------------------------------
# 6. Confirm frozen R4 policy still unchanged
# ----------------------------------------------------------------------------------------------------

import hashlib

policy_path = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

EXPECTED_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

actual_sha = hashlib.sha256(
    policy_path.read_bytes()
).hexdigest()

print("\n" + "=" * 110)
print("FROZEN R4 POLICY INTEGRITY")
print("=" * 110)

print("Expected:", EXPECTED_SHA)
print("Actual  :", actual_sha)

assert actual_sha == EXPECTED_SHA

print("\nPASS — frozen R4 policy remains unchanged")
print("PASS — robustness asset audit complete")

R4-17 — ROBUSTNESS ASSET AND DATASET AUDIT

DATASET / TABLE FILES
--------------------------------------------------------------------------------------------------------------
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv          302 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_test_split.csv              81,369 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen_oof_mass_policy.json               866 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_feasible_mass_quantiles.csv           229 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_oof_training_calibration_rows.csv      391,762 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction.json          1,033 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction_cv.csv          305 bytes
XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_training_calibration_rows.csv    

In [162]:
# ====================================================================================================
# R4-18 — FROZEN MODEL RECONSTRUCTION FEASIBILITY AUDIT
# ====================================================================================================

from pathlib import Path
import pandas as pd
import json
import hashlib

print("=" * 110)
print("R4-18 — FROZEN MODEL RECONSTRUCTION FEASIBILITY AUDIT")
print("=" * 110)

ROOT = Path(
    "/content/R4_work/recovered_R3/"
    "XAgentic_R3_Candidate_Masking_FINAL"
)

R4_OUT = Path("/content/R4_work/R4_results")

# ----------------------------------------------------------------------------------------------------
# 1. Important provenance files
# ----------------------------------------------------------------------------------------------------

files = {
    "cv":
        ROOT / "02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv",

    "seed_reproduction":
        ROOT / "02_PROVENANCE/R3_seed42_model_reproduction.json",

    "seed_reproduction_cv":
        ROOT / "02_PROVENANCE/R3_seed42_model_reproduction_cv.csv",

    "root_reproduction":
        ROOT / "R3_model_reproduction.json",

    "root_reproduction_cv":
        ROOT / "R3_model_reproduction_cv.csv",

    "training":
        ROOT / "02_PROVENANCE/R3_training_calibration_rows.csv",

    "oof_training":
        ROOT / "02_PROVENANCE/R3_oof_training_calibration_rows.csv",

    "test":
        ROOT / "02_PROVENANCE/R3_frozen1200_test_split.csv",
}

print("\nPROVENANCE FILE CHECK")
print("-" * 110)

for name, p in files.items():

    print(
        f"{name:25s}",
        "PASS" if p.exists() else "MISSING",
        p
    )

# ----------------------------------------------------------------------------------------------------
# 2. Show CV model ranking
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FROZEN MODEL CV RESULTS")
print("=" * 110)

cv = pd.read_csv(files["cv"])

print(cv.to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# 3. Inspect reproduction JSON files
# ----------------------------------------------------------------------------------------------------

for key in [
    "seed_reproduction",
    "root_reproduction",
]:

    p = files[key]

    print("\n" + "=" * 110)
    print(key.upper())
    print("=" * 110)

    with open(p, "r") as f:
        obj = json.load(f)

    print(
        json.dumps(
            obj,
            indent=2,
            sort_keys=True
        )
    )

# ----------------------------------------------------------------------------------------------------
# 4. Inspect reproduction CV tables
# ----------------------------------------------------------------------------------------------------

for key in [
    "seed_reproduction_cv",
    "root_reproduction_cv",
]:

    p = files[key]

    print("\n" + "=" * 110)
    print(key.upper())
    print("=" * 110)

    x = pd.read_csv(p)

    print(x.to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# 5. Verify training/test partition
# ----------------------------------------------------------------------------------------------------

train = pd.read_csv(files["training"])
test = pd.read_csv(files["test"])

print("\n" + "=" * 110)
print("TRAIN / TEST PARTITION")
print("=" * 110)

print("Training rows :", len(train))
print("Test rows     :", len(test))
print("Total         :", len(train) + len(test))

train_ids = set(train["id"].astype(int))
test_ids = set(test["id"].astype(int))

overlap = train_ids & test_ids

print("ID overlap    :", len(overlap))

assert len(train) == 840
assert len(test) == 360
assert len(overlap) == 0

print("PASS — frozen 840/360 partition intact")

# ----------------------------------------------------------------------------------------------------
# 6. Determine what fields are available for model reconstruction
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("TRAINING FIELDS AVAILABLE")
print("=" * 110)

for c in train.columns:
    print(c)

required_basic = [
    "text",
    "truth",
]

for c in required_basic:
    assert c in train.columns

print("\nPASS — text and target labels available")

# ----------------------------------------------------------------------------------------------------
# 7. Compare stored training predictions/probabilities
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("STORED MODEL OUTPUTS")
print("=" * 110)

prob_cols = [
    "p_eMBB",
    "p_URLLC",
    "p_mMTC",
]

for c in prob_cols:
    assert c in train.columns
    assert c in test.columns

print("Training probabilities available : YES")
print("Test probabilities available     : YES")

print("\nTraining baseline accuracy:")
print(
    (
        train["baseline_pred"].astype(str) ==
        train["truth"].astype(str)
    ).mean()
)

print("\nTest baseline accuracy:")
print(
    (
        test["baseline_pred"].astype(str) ==
        test["truth"].astype(str)
    ).mean()
)

# ----------------------------------------------------------------------------------------------------
# 8. Search reproduction metadata for likely model/vectorizer settings
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("RECONSTRUCTION INFORMATION SUMMARY")
print("=" * 110)

print("""
We need the provenance above to answer:

1. Which classifier was frozen?
2. Which text vectorizer was used?
3. Which random seed was used?
4. Which hyperparameters were used?
5. Can training on the frozen 840 rows reproduce the exact
   360 held-out predictions and probabilities?
""")

# ----------------------------------------------------------------------------------------------------
# 9. Preserve current R4 policy integrity
# ----------------------------------------------------------------------------------------------------

policy_path = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

EXPECTED_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

actual_sha = hashlib.sha256(
    policy_path.read_bytes()
).hexdigest()

assert actual_sha == EXPECTED_SHA

print("\nFrozen R4 policy SHA256:")
print(actual_sha)

print("\nPASS — R4-18 reconstruction feasibility audit complete")
print("PASS — frozen R4 policy unchanged")

R4-18 — FROZEN MODEL RECONSTRUCTION FEASIBILITY AUDIT

PROVENANCE FILE CHECK
--------------------------------------------------------------------------------------------------------------
cv                        PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_frozen1200_selection_agent_cv.csv
seed_reproduction         PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction.json
seed_reproduction_cv      PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_seed42_model_reproduction_cv.csv
root_reproduction         PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/R3_model_reproduction.json
root_reproduction_cv      PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/R3_model_reproduction_cv.csv
training                  PASS /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/02_PROVENANCE/R3_training

In [163]:
# ====================================================================================================
# R4-19 — SEARCH COLAB FOR ORIGINAL EXECUTABLE SELECTION-AGENT PIPELINE
# ====================================================================================================

from pathlib import Path
import hashlib
import json

print("=" * 110)
print("R4-19 — SEARCH COLAB FOR ORIGINAL EXECUTABLE SELECTION-AGENT PIPELINE")
print("=" * 110)

SEARCH_ROOTS = [
    Path("/content"),
]

TARGET_NAMES = [
    "xagentic_common.py",
    "R2_strict_feasibility_oracle_v2.py",
]

# ----------------------------------------------------------------------------------------------------
# 1. Search exact filenames
# ----------------------------------------------------------------------------------------------------

found = {}

for target in TARGET_NAMES:

    found[target] = []

    for root in SEARCH_ROOTS:

        try:
            matches = list(root.rglob(target))
        except Exception:
            matches = []

        for p in matches:

            try:
                sha = hashlib.sha256(
                    p.read_bytes()
                ).hexdigest()

                found[target].append(
                    {
                        "path": str(p),
                        "size": p.stat().st_size,
                        "sha256": sha,
                    }
                )

            except Exception as e:

                found[target].append(
                    {
                        "path": str(p),
                        "error": repr(e),
                    }
                )

# ----------------------------------------------------------------------------------------------------
# 2. Print results
# ----------------------------------------------------------------------------------------------------

for target in TARGET_NAMES:

    print("\n" + "=" * 110)
    print(target)
    print("=" * 110)

    rows = found[target]

    if not rows:
        print("NOT FOUND")
        continue

    for x in rows:

        print("\nPATH:")
        print(x["path"])

        if "size" in x:
            print("SIZE   :", f'{x["size"]:,}', "bytes")
            print("SHA256 :", x["sha256"])
        else:
            print("ERROR  :", x["error"])

# ----------------------------------------------------------------------------------------------------
# 3. Check specifically for the expected historical SHA
# ----------------------------------------------------------------------------------------------------

EXPECTED_COMMON_SHA = (
    "1484f474ba7f40b451583115da9fc7c4"
    "fdebc79f9c6876d9a0d7447f649cfbb6"
)

EXPECTED_ORACLE_SHA = (
    "75b1cd0e4847f2cbc38690ec32c1b0d"
    "d17343c89582e0f4ed206fb7421867cd1"
)

print("\n" + "=" * 110)
print("EXPECTED SHA MATCHES")
print("=" * 110)

common_matches = [
    x for x in found["xagentic_common.py"]
    if x.get("sha256") == EXPECTED_COMMON_SHA
]

oracle_matches = [
    x for x in found["R2_strict_feasibility_oracle_v2.py"]
    if x.get("sha256") == EXPECTED_ORACLE_SHA
]

print(
    "Exact xagentic_common.py matches :",
    len(common_matches)
)

for x in common_matches:
    print("  ", x["path"])

print(
    "\nExact R2 oracle matches          :",
    len(oracle_matches)
)

for x in oracle_matches:
    print("  ", x["path"])

# ----------------------------------------------------------------------------------------------------
# 4. Search more broadly for Python source files with useful names
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("OTHER POSSIBLY RELEVANT PYTHON FILES")
print("=" * 110)

keywords = [
    "xagentic",
    "selection",
    "experiment",
    "source_analyzer",
    "feasibility",
    "candidate",
]

possible = []

for root in SEARCH_ROOTS:

    for p in root.rglob("*.py"):

        s = str(p).lower()

        if any(k in s for k in keywords):

            try:
                possible.append(
                    (
                        str(p),
                        p.stat().st_size
                    )
                )
            except:
                pass

# Deduplicate
possible = sorted(set(possible))

print("Count:", len(possible))

for path, size in possible[:200]:
    print(f"{size:>10,}  {path}")

if len(possible) > 200:
    print(
        f"\n... {len(possible)-200} additional files omitted"
    )

# ----------------------------------------------------------------------------------------------------
# 5. Preserve search evidence
# ----------------------------------------------------------------------------------------------------

R4_OUT = Path("/content/R4_work/R4_results")

search_evidence = {
    "xagentic_common_expected_sha256":
        EXPECTED_COMMON_SHA,

    "r2_oracle_expected_sha256":
        EXPECTED_ORACLE_SHA,

    "exact_xagentic_common_matches":
        common_matches,

    "exact_r2_oracle_matches":
        oracle_matches,

    "all_exact_filename_search_results":
        found,
}

out = (
    R4_OUT /
    "R4_source_recovery_search.json"
)

with open(out, "w") as f:
    json.dump(
        search_evidence,
        f,
        indent=2,
        sort_keys=True
    )

print("\nEvidence written:")
print(out)

# ----------------------------------------------------------------------------------------------------
# 6. Verify R4 policy remains untouched
# ----------------------------------------------------------------------------------------------------

policy = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

EXPECTED_R4_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

actual = hashlib.sha256(
    policy.read_bytes()
).hexdigest()

assert actual == EXPECTED_R4_SHA

print("\nR4 policy SHA256:")
print(actual)

print("\nPASS — source recovery search complete")
print("PASS — frozen R4 policy unchanged")

R4-19 — SEARCH COLAB FOR ORIGINAL EXECUTABLE SELECTION-AGENT PIPELINE

xagentic_common.py

PATH:
/content/R2_FROZEN_REFERENCE/XAgentic_R2_Strict_Feasibility_FINAL/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_common.py
SIZE   : 26,471 bytes
SHA256 : 1484f474ba7f40b451583115da9fc7c4fdebc79f9c6876d9a0d7447f649cfbb6

PATH:
/content/xagentic_r1_work/XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_common.py
SIZE   : 26,471 bytes
SHA256 : 1484f474ba7f40b451583115da9fc7c4fdebc79f9c6876d9a0d7447f649cfbb6

PATH:
/content/xagentic_r2_work/01_SOURCE/BASE_E1_E5_EXTRACTED/XAgentic_Final_Paper_Evidence_2026-10-06/03_SOURCE_CODE/xagentic_common.py
SIZE   : 26,471 bytes
SHA256 : 1484f474ba7f40b451583115da9fc7c4fdebc79f9c6876d9a0d7447f649cfbb6

R2_strict_feasibility_oracle_v2.py

PATH:
/content/R2_FROZEN_REFERENCE/XAgentic_R2_Strict_Feasibility_FINAL/02_R2_AUDIT/R2_strict_feasibility_oracle_v2.py
SIZE   : 2,026 bytes
SHA256 : 75b1cd0e48

In [1]:
# ====================================================================================================
# R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION
# ====================================================================================================

from pathlib import Path
import hashlib
import re

print("=" * 110)
print("R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION")
print("=" * 110)

SRC = Path(
    "/content/R2_FROZEN_REFERENCE/"
    "XAgentic_R2_Strict_Feasibility_FINAL/"
    "01_SOURCE/BASE_E1_E5_EXTRACTED/"
    "XAgentic_Final_Paper_Evidence_2026-10-06/"
    "03_SOURCE_CODE/xagentic_common.py"
)

EXPECTED_SHA = (
    "1484f474ba7f40b451583115da9fc7c4"
    "fdebc79f9c6876d9a0d7447f649cfbb6"
)

assert SRC.exists()

actual_sha = hashlib.sha256(
    SRC.read_bytes()
).hexdigest()

assert actual_sha == EXPECTED_SHA

print("\nSOURCE")
print("-" * 110)
print("Path   :", SRC)
print("Bytes  :", SRC.stat().st_size)
print("SHA256 :", actual_sha)

text = SRC.read_text(
    encoding="utf-8",
    errors="replace"
)

lines = text.splitlines()

# ----------------------------------------------------------------------------------------------------
# 1. Imports
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("IMPORTS")
print("=" * 110)

for i, line in enumerate(lines, start=1):
    s = line.strip()

    if s.startswith("import ") or s.startswith("from "):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 2. Functions / classes
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FUNCTIONS / CLASSES")
print("=" * 110)

for i, line in enumerate(lines, start=1):

    s = line.strip()

    if (
        s.startswith("def ") or
        s.startswith("class ")
    ):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 3. Search for model / feature construction
# ----------------------------------------------------------------------------------------------------

patterns = [
    "GradientBoosting",
    "RandomForest",
    "DecisionTree",
    "LogisticRegression",
    "MLPClassifier",
    "Tfidf",
    "CountVectorizer",
    "vectorizer",
    "Pipeline",
    "FeatureUnion",
    "train_test_split",
    "StratifiedKFold",
    "cross_val",
    "predict_proba",
    "fit(",
    "random_state",
    "SEED",
    "seed",
]

print("\n" + "=" * 110)
print("MODEL / FEATURE / TRAINING REFERENCES")
print("=" * 110)

hits = []

for i, line in enumerate(lines, start=1):

    if any(
        p.lower() in line.lower()
        for p in patterns
    ):
        hits.append(i)

# Print context around hits
shown = set()

for hit in hits:

    start = max(1, hit - 3)
    end = min(len(lines), hit + 5)

    for j in range(start, end + 1):

        if j not in shown:
            print(f"{j:4d}: {lines[j-1]}")
            shown.add(j)

    print("-" * 110)

# ----------------------------------------------------------------------------------------------------
# 4. Search for slice-label mappings
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SLICE / LABEL REFERENCES")
print("=" * 110)

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(
        k in lower
        for k in [
            "embb",
            "urllc",
            "mmtc",
            "slice_id",
            "slices",
        ]
    ):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 5. Search for dataset generation / text construction
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("DATASET / TEXT GENERATION REFERENCES")
print("=" * 110)

dataset_terms = [
    "generate",
    "dataset",
    "corpus",
    "template",
    "text",
    "intent",
    "utterance",
]

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(term in lower for term in dataset_terms):

        if (
            "def " in line or
            "return " in line or
            "template" in lower or
            "dataset" in lower or
            "corpus" in lower
        ):
            print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 6. Preserve exact source in R4 evidence
# ----------------------------------------------------------------------------------------------------

R4_OUT = Path("/content/R4_work/R4_results")

audit_file = R4_OUT / "R4_selection_agent_source_audit.txt"

with open(audit_file, "w") as f:

    f.write(
        "R4 EXACT SELECTION-AGENT SOURCE AUDIT\n"
    )

    f.write("=" * 80 + "\n")

    f.write(f"Source: {SRC}\n")
    f.write(f"SHA256: {actual_sha}\n")
    f.write(f"Bytes: {SRC.stat().st_size}\n")

print("\nAudit evidence:")
print(audit_file)

# ----------------------------------------------------------------------------------------------------
# 7. Frozen R4 policy integrity
# ----------------------------------------------------------------------------------------------------

POLICY = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

EXPECTED_R4_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

r4_sha = hashlib.sha256(
    POLICY.read_bytes()
).hexdigest()

assert r4_sha == EXPECTED_R4_SHA

print("\nFrozen R4 policy SHA256:")
print(r4_sha)

print("\nPASS — exact selection-agent source inspected")
print("PASS — frozen R4 policy unchanged")

R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION


AssertionError: 

In [2]:
# ====================================================================================================
# R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION — PATH-SAFE VERSION
# ====================================================================================================

from pathlib import Path
import hashlib

print("=" * 110)
print("R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION")
print("=" * 110)

EXPECTED_SHA = (
    "1484f474ba7f40b451583115da9fc7c4"
    "fdebc79f9c6876d9a0d7447f649cfbb6"
)

# ----------------------------------------------------------------------------------------------------
# 1. Locate exact SHA-matching xagentic_common.py
# ----------------------------------------------------------------------------------------------------

print("\nSEARCHING FOR EXACT SOURCE")
print("-" * 110)

candidates = list(
    Path("/content").rglob("xagentic_common.py")
)

print("Filename matches found:", len(candidates))

exact_matches = []

for p in candidates:

    try:
        sha = hashlib.sha256(
            p.read_bytes()
        ).hexdigest()

        print("\nPATH:")
        print(p)
        print("SHA256:", sha)

        if sha == EXPECTED_SHA:
            exact_matches.append(p)

    except Exception as e:
        print("READ ERROR:", p, repr(e))

assert exact_matches, (
    "STOP: no SHA-matching xagentic_common.py found"
)

# Prefer the R2 frozen reference copy if it still exists.
preferred = [
    p for p in exact_matches
    if "R2_FROZEN_REFERENCE" in str(p)
]

SRC = (
    preferred[0]
    if preferred
    else exact_matches[0]
)

print("\n" + "=" * 110)
print("SELECTED EXACT SOURCE")
print("=" * 110)

print("Path   :", SRC)
print("Bytes  :", SRC.stat().st_size)

actual_sha = hashlib.sha256(
    SRC.read_bytes()
).hexdigest()

print("SHA256 :", actual_sha)

assert actual_sha == EXPECTED_SHA

# ----------------------------------------------------------------------------------------------------
# 2. Read source
# ----------------------------------------------------------------------------------------------------

text = SRC.read_text(
    encoding="utf-8",
    errors="replace"
)

lines = text.splitlines()

print("Lines  :", len(lines))

# ----------------------------------------------------------------------------------------------------
# 3. Imports
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("IMPORTS")
print("=" * 110)

for i, line in enumerate(lines, start=1):

    s = line.strip()

    if (
        s.startswith("import ") or
        s.startswith("from ")
    ):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 4. Functions and classes
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("FUNCTIONS / CLASSES")
print("=" * 110)

for i, line in enumerate(lines, start=1):

    s = line.strip()

    if (
        s.startswith("def ") or
        s.startswith("class ")
    ):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 5. Model / feature / training references
# ----------------------------------------------------------------------------------------------------

patterns = [
    "GradientBoosting",
    "GradientBoostingClassifier",
    "RandomForest",
    "RandomForestClassifier",
    "DecisionTree",
    "DecisionTreeClassifier",
    "LogisticRegression",
    "MLPClassifier",
    "Tfidf",
    "TfidfVectorizer",
    "CountVectorizer",
    "vectorizer",
    "Pipeline",
    "FeatureUnion",
    "train_test_split",
    "StratifiedKFold",
    "cross_val",
    "predict_proba",
    ".fit(",
    "random_state",
    "SEED",
    "seed",
]

print("\n" + "=" * 110)
print("MODEL / FEATURE / TRAINING REFERENCES")
print("=" * 110)

hits = []

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(
        p.lower() in lower
        for p in patterns
    ):
        hits.append(i)

shown = set()

for hit in hits:

    start = max(1, hit - 4)
    end = min(len(lines), hit + 6)

    for j in range(start, end + 1):

        if j not in shown:
            print(f"{j:4d}: {lines[j-1]}")
            shown.add(j)

    print("-" * 110)

# ----------------------------------------------------------------------------------------------------
# 6. Slice / label mappings
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SLICE / LABEL REFERENCES")
print("=" * 110)

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(
        token in lower
        for token in [
            "embb",
            "urllc",
            "mmtc",
            "slice_id",
            "slices",
        ]
    ):
        print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 7. Dataset / text generation references
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("DATASET / TEXT GENERATION REFERENCES")
print("=" * 110)

terms = [
    "generate",
    "dataset",
    "corpus",
    "template",
    "utterance",
]

for i, line in enumerate(lines, start=1):

    lower = line.lower()

    if any(term in lower for term in terms):

        if (
            "def " in lower or
            "return " in lower or
            "template" in lower or
            "dataset" in lower or
            "corpus" in lower
        ):
            print(f"{i:4d}: {line}")

# ----------------------------------------------------------------------------------------------------
# 8. Explicitly show likely model-building functions in full
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("LIKELY MODEL-BUILDING FUNCTION BLOCKS")
print("=" * 110)

interesting_function_terms = [
    "model",
    "train",
    "selection",
    "classifier",
    "feature",
]

function_starts = []

for i, line in enumerate(lines):

    stripped = line.strip()

    if stripped.startswith("def "):

        name = stripped.split("def ", 1)[1].split("(", 1)[0]

        if any(
            term in name.lower()
            for term in interesting_function_terms
        ):
            function_starts.append((i, name))

for start_idx, name in function_starts:

    # Locate next top-level function/class.
    end_idx = len(lines)

    for j in range(start_idx + 1, len(lines)):

        candidate = lines[j]

        if (
            candidate.startswith("def ") or
            candidate.startswith("class ")
        ):
            end_idx = j
            break

    print("\n" + "-" * 110)
    print("FUNCTION:", name)
    print("-" * 110)

    for j in range(start_idx, end_idx):
        print(f"{j+1:4d}: {lines[j]}")

# ----------------------------------------------------------------------------------------------------
# 9. Save source audit evidence
# ----------------------------------------------------------------------------------------------------

R4_OUT = Path("/content/R4_work/R4_results")
R4_OUT.mkdir(parents=True, exist_ok=True)

AUDIT = (
    R4_OUT /
    "R4_selection_agent_source_audit.txt"
)

with open(AUDIT, "w", encoding="utf-8") as f:

    f.write(
        "R4 EXACT SELECTION-AGENT SOURCE AUDIT\n"
    )

    f.write("=" * 100 + "\n")

    f.write(
        f"Selected source: {SRC}\n"
    )

    f.write(
        f"Source bytes: {SRC.stat().st_size}\n"
    )

    f.write(
        f"Source SHA256: {actual_sha}\n"
    )

    f.write(
        f"Exact matching copies found: "
        f"{len(exact_matches)}\n"
    )

    f.write("\nExact copies:\n")

    for p in exact_matches:
        f.write(f"- {p}\n")

print("\nAudit evidence:")
print(AUDIT)

# ----------------------------------------------------------------------------------------------------
# 10. Verify frozen R4 policy
# ----------------------------------------------------------------------------------------------------

POLICY = (
    R4_OUT /
    "R4_frozen_uncertainty_policy.json"
)

EXPECTED_R4_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

assert POLICY.exists()

r4_sha = hashlib.sha256(
    POLICY.read_bytes()
).hexdigest()

print("\n" + "=" * 110)
print("FROZEN R4 POLICY INTEGRITY")
print("=" * 110)

print("Expected:", EXPECTED_R4_SHA)
print("Actual  :", r4_sha)

assert r4_sha == EXPECTED_R4_SHA

print("\nPASS — exact SHA-matching selection-agent source inspected")
print("PASS — source audit evidence saved")
print("PASS — frozen R4 policy unchanged")

R4-20 — INSPECT EXACT SELECTION-AGENT IMPLEMENTATION

SEARCHING FOR EXACT SOURCE
--------------------------------------------------------------------------------------------------------------
Filename matches found: 0


AssertionError: STOP: no SHA-matching xagentic_common.py found

In [3]:
# ====================================================================================================
# R4-20A — COLAB RUNTIME RECOVERY AUDIT
# ====================================================================================================

from pathlib import Path
import os
import hashlib

print("=" * 110)
print("R4-20A — COLAB RUNTIME RECOVERY AUDIT")
print("=" * 110)

# ----------------------------------------------------------------------------------------------------
# 1. Current /content state
# ----------------------------------------------------------------------------------------------------

print("\n/content TOP-LEVEL CONTENTS")
print("-" * 110)

content = Path("/content")

for p in sorted(content.iterdir()):
    try:
        kind = "DIR " if p.is_dir() else "FILE"
        print(f"{kind:4s}  {p}")
    except:
        pass

# ----------------------------------------------------------------------------------------------------
# 2. Important R4 paths
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("R4 WORKSPACE CHECK")
print("=" * 110)

important = [
    Path("/content/R4_work"),
    Path("/content/R4_work/R4_results"),
    Path("/content/R4_work/recovered_R3"),
    Path("/content/R4_work/R4_results/R4_frozen_uncertainty_policy.json"),
    Path("/content/R4_work/R4_results/R4_frozen_policy_heldout_decisions.csv"),
    Path("/content/R4_work/R4_results/R4_frozen_policy_heldout_summary.json"),
]

for p in important:
    print(
        "PASS   " if p.exists() else "MISSING",
        p
    )

# ----------------------------------------------------------------------------------------------------
# 3. Verify R4 frozen policy if available
# ----------------------------------------------------------------------------------------------------

EXPECTED_R4_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

policy = Path(
    "/content/R4_work/R4_results/"
    "R4_frozen_uncertainty_policy.json"
)

print("\n" + "=" * 110)
print("FROZEN R4 POLICY")
print("=" * 110)

if policy.exists():

    sha = hashlib.sha256(
        policy.read_bytes()
    ).hexdigest()

    print("Expected:", EXPECTED_R4_SHA)
    print("Actual  :", sha)

    assert sha == EXPECTED_R4_SHA

    print("PASS — frozen R4 policy intact")

else:

    print("MISSING — frozen R4 policy is not currently present")

# ----------------------------------------------------------------------------------------------------
# 4. Search for R1/R2/R3/R4 directories
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("EXPERIMENT DIRECTORY SEARCH")
print("=" * 110)

tokens = [
    "r1",
    "r2",
    "r3",
    "r4",
    "xagentic",
]

dirs = []

for p in Path("/content").rglob("*"):

    try:
        if p.is_dir():

            s = str(p).lower()

            if any(t in s for t in tokens):
                dirs.append(p)

    except:
        pass

print("Matching directories:", len(dirs))

for p in sorted(dirs)[:150]:
    print(p)

if len(dirs) > 150:
    print(f"... {len(dirs)-150} additional directories")

# ----------------------------------------------------------------------------------------------------
# 5. Search for any surviving ZIPs
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("ZIP FILES CURRENTLY IN /content")
print("=" * 110)

zips = list(
    Path("/content").rglob("*.zip")
)

print("ZIP count:", len(zips))

for p in sorted(zips):

    try:
        print(
            f"{p.stat().st_size:>12,} bytes  {p}"
        )
    except:
        print(p)

# ----------------------------------------------------------------------------------------------------
# 6. Search exact source name again
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("SOURCE SEARCH")
print("=" * 110)

sources = list(
    Path("/content").rglob("xagentic_common.py")
)

print("xagentic_common.py count:", len(sources))

for p in sources:

    sha = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    print("\n", p)
    print(" SHA256:", sha)

print("\n" + "=" * 110)
print("R4-20A COMPLETE")
print("=" * 110)

R4-20A — COLAB RUNTIME RECOVERY AUDIT

/content TOP-LEVEL CONTENTS
--------------------------------------------------------------------------------------------------------------
DIR   /content/.config
DIR   /content/sample_data

R4 WORKSPACE CHECK
MISSING /content/R4_work
MISSING /content/R4_work/R4_results
MISSING /content/R4_work/recovered_R3
MISSING /content/R4_work/R4_results/R4_frozen_uncertainty_policy.json
MISSING /content/R4_work/R4_results/R4_frozen_policy_heldout_decisions.csv
MISSING /content/R4_work/R4_results/R4_frozen_policy_heldout_summary.json

FROZEN R4 POLICY
MISSING — frozen R4 policy is not currently present

EXPERIMENT DIRECTORY SEARCH
Matching directories: 0

ZIP FILES CURRENTLY IN /content
ZIP count: 0

SOURCE SEARCH
xagentic_common.py count: 0

R4-20A COMPLETE


In [4]:
# ====================================================================================================
# R4-RECOVERY-01 — UPLOAD FROZEN R3 FINAL PACKAGE AFTER COLAB RESET
# ====================================================================================================

from google.colab import files
from pathlib import Path
import hashlib

print("=" * 110)
print("R4-RECOVERY-01 — UPLOAD FROZEN R3 FINAL PACKAGE")
print("=" * 110)

uploaded = files.upload()

assert uploaded, "STOP: no file uploaded"

print("\nUPLOADED FILES")
print("-" * 110)

for name, data in uploaded.items():

    p = Path("/content") / name

    sha = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    print("\nFile   :", name)
    print("Bytes  :", f"{p.stat().st_size:,}")
    print("SHA256 :", sha)

print("\nUpload complete.")

R4-RECOVERY-01 — UPLOAD FROZEN R3 FINAL PACKAGE


Saving XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip to XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip

UPLOADED FILES
--------------------------------------------------------------------------------------------------------------

File   : XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip
Bytes  : 329,177
SHA256 : ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152

Upload complete.


In [5]:
# ====================================================================================================
# R4-RECOVERY-02 — RESTORE AND VERIFY FROZEN R3 EVIDENCE PACKAGE
# ====================================================================================================

from pathlib import Path
import zipfile
import hashlib
import pandas as pd
import shutil

print("=" * 110)
print("R4-RECOVERY-02 — RESTORE AND VERIFY FROZEN R3 EVIDENCE PACKAGE")
print("=" * 110)

ZIP = Path(
    "/content/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip"
)

EXPECTED_ZIP_SHA = (
    "ba0f680c372ddda498eba2f8b84f5df9"
    "a599dc04810a679b53ce77af6fce0152"
)

WORK = Path("/content/R4_work")
RECOVERED = WORK / "recovered_R3"
RESULTS = WORK / "R4_results"

# ----------------------------------------------------------------------------------------------------
# 1. Verify uploaded ZIP again
# ----------------------------------------------------------------------------------------------------

assert ZIP.exists(), f"STOP: missing {ZIP}"

zip_sha = hashlib.sha256(
    ZIP.read_bytes()
).hexdigest()

print("\nZIP INTEGRITY")
print("-" * 110)

print("File    :", ZIP)
print("Bytes   :", f"{ZIP.stat().st_size:,}")
print("Expected:", EXPECTED_ZIP_SHA)
print("Actual  :", zip_sha)

assert zip_sha == EXPECTED_ZIP_SHA, (
    "STOP: uploaded R3 ZIP does not match frozen SHA256"
)

print("PASS — exact frozen R3 ZIP verified")

# ----------------------------------------------------------------------------------------------------
# 2. Create clean R4 recovery workspace
# ----------------------------------------------------------------------------------------------------

if WORK.exists():
    shutil.rmtree(WORK)

RECOVERED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

print("\nWORKSPACE")
print("-" * 110)

print("R4 workspace :", WORK)
print("Recovered R3 :", RECOVERED)
print("R4 results   :", RESULTS)

# ----------------------------------------------------------------------------------------------------
# 3. Extract R3 package
# ----------------------------------------------------------------------------------------------------

with zipfile.ZipFile(ZIP, "r") as z:
    members = z.namelist()

    print("\nZIP members:", len(members))

    z.extractall(RECOVERED)

print("PASS — R3 package extracted")

# ----------------------------------------------------------------------------------------------------
# 4. Show recovered top-level structure
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("RECOVERED R3 STRUCTURE")
print("=" * 110)

for p in sorted(RECOVERED.iterdir()):
    print(
        "DIR " if p.is_dir() else "FILE",
        p
    )

# ----------------------------------------------------------------------------------------------------
# 5. Locate manifest
# ----------------------------------------------------------------------------------------------------

manifest_matches = list(
    RECOVERED.rglob("R3_SHA256_MANIFEST.csv")
)

print("\n" + "=" * 110)
print("R3 MANIFEST")
print("=" * 110)

assert manifest_matches, (
    "STOP: R3_SHA256_MANIFEST.csv not found"
)

assert len(manifest_matches) == 1, (
    f"STOP: expected one manifest, found {len(manifest_matches)}"
)

MANIFEST = manifest_matches[0]

print("Manifest:")
print(MANIFEST)

manifest = pd.read_csv(MANIFEST)

print("Rows   :", len(manifest))
print("Columns:", list(manifest.columns))

print("\nFirst rows:")
print(manifest.head().to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# 6. Validate manifest dynamically
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("MANIFEST FILE VALIDATION")
print("=" * 110)

# Determine likely filename/path and SHA columns without assuming exact names.

lower_map = {
    c.lower(): c
    for c in manifest.columns
}

path_col = None
sha_col = None

for c in manifest.columns:
    lc = c.lower()

    if path_col is None and (
        "path" in lc or
        "file" in lc
    ):
        path_col = c

    if sha_col is None and (
        "sha256" in lc or
        "sha_256" in lc or
        lc == "sha"
    ):
        sha_col = c

print("Detected path column:", path_col)
print("Detected SHA column :", sha_col)

assert path_col is not None
assert sha_col is not None

# Manifest paths may be relative to the package root.
package_root = MANIFEST.parents[2]

print("Package root        :", package_root)

checks = []

for _, row in manifest.iterrows():

    rel = str(row[path_col]).strip()
    expected = str(row[sha_col]).strip().lower()

    candidates = [
        package_root / rel,
        RECOVERED / rel,
    ]

    target = None

    for c in candidates:
        if c.exists() and c.is_file():
            target = c
            break

    if target is None:
        # Last-resort basename lookup, but require uniqueness.
        matches = list(
            package_root.rglob(Path(rel).name)
        )

        if len(matches) == 1:
            target = matches[0]

    if target is None:

        checks.append({
            "file": rel,
            "status": "MISSING",
            "expected_sha256": expected,
            "actual_sha256": None,
        })

        continue

    actual = hashlib.sha256(
        target.read_bytes()
    ).hexdigest()

    status = (
        "PASS"
        if actual == expected
        else "FAILED"
    )

    checks.append({
        "file": rel,
        "status": status,
        "expected_sha256": expected,
        "actual_sha256": actual,
    })

audit = pd.DataFrame(checks)

print("\nStatus counts:")
print(
    audit["status"]
    .value_counts(dropna=False)
    .to_string()
)

failed = audit[
    audit["status"] != "PASS"
]

if len(failed):
    print("\nNON-PASS ROWS:")
    print(failed.to_string(index=False))

assert len(failed) == 0, (
    "STOP: recovered R3 internal evidence failed manifest verification"
)

print(
    f"\nPASS — all {len(audit)} manifest entries verified"
)

# ----------------------------------------------------------------------------------------------------
# 7. Locate critical R3 artifacts
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("CRITICAL R3 ARTIFACT CHECK")
print("=" * 110)

critical_names = [
    "R3_frozen1200_test_split.csv",
    "R3_oof_training_calibration_rows.csv",
    "R3_final_frozen_policy_heldout_decisions.csv",
    "R3_final_frozen_policy_summary.json",
    "R3_final_system_comparison.csv",
]

critical = {}

for name in critical_names:

    matches = list(
        RECOVERED.rglob(name)
    )

    if len(matches) == 1:
        critical[name] = matches[0]

        print(
            "PASS ",
            name,
            "->",
            matches[0]
        )

    elif len(matches) == 0:
        print("MISSING", name)

    else:
        print(
            "MULTIPLE",
            name,
            len(matches)
        )

assert all(
    name in critical
    for name in critical_names
), "STOP: one or more critical R3 artifacts missing"

# ----------------------------------------------------------------------------------------------------
# 8. Verify expected 840 / 360 evidence sizes
# ----------------------------------------------------------------------------------------------------

test = pd.read_csv(
    critical["R3_frozen1200_test_split.csv"]
)

oof = pd.read_csv(
    critical["R3_oof_training_calibration_rows.csv"]
)

held = pd.read_csv(
    critical["R3_final_frozen_policy_heldout_decisions.csv"]
)

print("\n" + "=" * 110)
print("R3 DATASET SIZE CHECK")
print("=" * 110)

print("OOF development rows :", len(oof))
print("Frozen test rows     :", len(test))
print("Heldout decisions    :", len(held))

assert len(oof) == 840
assert len(test) == 360
assert len(held) == 360

assert test["id"].nunique() == 360
assert held["id"].nunique() == 360

print("PASS — expected 840/360 split restored")

# ----------------------------------------------------------------------------------------------------
# 9. Save recovery audit
# ----------------------------------------------------------------------------------------------------

audit_out = (
    RESULTS /
    "R4_recovered_R3_manifest_audit.csv"
)

audit.to_csv(
    audit_out,
    index=False
)

recovery_info = pd.DataFrame([
    {
        "r3_zip": ZIP.name,
        "r3_zip_bytes": ZIP.stat().st_size,
        "r3_zip_sha256": zip_sha,
        "manifest_entries_verified": len(audit),
        "oof_rows": len(oof),
        "test_rows": len(test),
        "heldout_rows": len(held),
        "status": "PASS",
    }
])

recovery_info.to_csv(
    RESULTS /
    "R4_runtime_recovery_summary.csv",
    index=False
)

print("\nRecovery evidence written:")
print(audit_out)
print(
    RESULTS /
    "R4_runtime_recovery_summary.csv"
)

print("\n" + "=" * 110)
print("PASS — FROZEN R3 PACKAGE FULLY RESTORED")
print("PASS — R4 RECOVERY WORKSPACE READY")
print("=" * 110)

R4-RECOVERY-02 — RESTORE AND VERIFY FROZEN R3 EVIDENCE PACKAGE

ZIP INTEGRITY
--------------------------------------------------------------------------------------------------------------
File    : /content/XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip
Bytes   : 329,177
Expected: ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152
Actual  : ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152
PASS — exact frozen R3 ZIP verified

WORKSPACE
--------------------------------------------------------------------------------------------------------------
R4 workspace : /content/R4_work
Recovered R3 : /content/R4_work/recovered_R3
R4 results   : /content/R4_work/R4_results

ZIP members: 30
PASS — R3 package extracted

RECOVERED R3 STRUCTURE
DIR  /content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL

R3 MANIFEST
Manifest:
/content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL/06_FINAL_MANIFEST/R3_SHA256_MANIFEST.csv
Rows   : 29
Columns: ['re

In [6]:
# ====================================================================================================
# R4-RECOVERY-03 — RECONSTRUCT ESSENTIAL R4 EVIDENCE FROM COMPLETED FROZEN RUN
# ====================================================================================================

from pathlib import Path
import pandas as pd
import json
import hashlib
from datetime import datetime

print("=" * 110)
print("R4-RECOVERY-03 — RECONSTRUCT ESSENTIAL R4 EVIDENCE")
print("=" * 110)

ROOT = Path("/content/R4_work/recovered_R3/XAgentic_R3_Candidate_Masking_FINAL")
OUT  = Path("/content/R4_work/R4_results")
OUT.mkdir(parents=True, exist_ok=True)

# ----------------------------------------------------------------------------------------------------
# 1. Immutable facts from completed R4 run
# ----------------------------------------------------------------------------------------------------

R3_TAU = 0.5486315987262682
R4_MARGIN_TAU = 0.7097637192002952

ORIGINAL_R4_POLICY_SHA = (
    "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42"
)

ORIGINAL_R4_HELDOUT_DECISIONS_SHA = (
    "8f7aab8ed6cdf66a5616d750f070ea02614c0fd1f3e0bf043f36d07f7527611a"
)

ORIGINAL_R4_HELDOUT_SUMMARY_SHA = (
    "9c4a1c98cf2d17da30045f60a5021b64bdd6deb410f3d7a310b207dff19ad412"
)

# ----------------------------------------------------------------------------------------------------
# 2. Load frozen R3 heldout decisions
# ----------------------------------------------------------------------------------------------------

r3_path = (
    ROOT /
    "03_RESULTS_RAW/R3_final_frozen_policy_heldout_decisions.csv"
)

r3 = pd.read_csv(r3_path)

assert len(r3) == 360
assert r3["id"].nunique() == 360

print("\nFrozen R3 heldout rows:", len(r3))

# ----------------------------------------------------------------------------------------------------
# 3. Reconstruct R4 uncertainty signals from frozen probabilities
# ----------------------------------------------------------------------------------------------------

prob_cols = ["p_eMBB", "p_URLLC", "p_mMTC"]

def uncertainty(row):
    vals = sorted(
        [float(row[c]) for c in prob_cols],
        reverse=True
    )
    top1 = vals[0]
    top2 = vals[1]
    margin = top1 - top2
    return pd.Series([top1, top2, margin])

r4 = r3.copy()

r4[
    [
        "r4_top1_confidence",
        "r4_top2_confidence",
        "r4_margin"
    ]
] = r4.apply(
    uncertainty,
    axis=1
)

# R3 autonomous state
r4["r4_r3_autonomous"] = (
    r4["final_action"].astype(str).str.upper() == "AUTONOMOUS"
)

r4["r4_r3_wrong"] = (
    r4["r4_r3_autonomous"] &
    (r4["final_pred"].astype(str) != r4["truth"].astype(str))
)

# Frozen R4 rule:
# only cases R3 would automate are eligible;
# lower margin than frozen threshold => uncertainty escalation.

r4["r4_uncertainty_escalated"] = (
    r4["r4_r3_autonomous"] &
    (r4["r4_margin"] < R4_MARGIN_TAU)
)

r4["r4_final_escalated"] = (
    (~r4["r4_r3_autonomous"]) |
    r4["r4_uncertainty_escalated"]
)

r4["r4_final_autonomous"] = ~r4["r4_final_escalated"]

r4["r4_final_correct"] = (
    r4["r4_final_autonomous"] &
    (r4["final_pred"].astype(str) == r4["truth"].astype(str))
)

r4["r4_final_wrong"] = (
    r4["r4_final_autonomous"] &
    (r4["final_pred"].astype(str) != r4["truth"].astype(str))
)

# ----------------------------------------------------------------------------------------------------
# 4. Verify reconstruction against completed R4 results
# ----------------------------------------------------------------------------------------------------

n = len(r4)

r3_esc = int((~r4["r4_r3_autonomous"]).sum())
r3_auto = int(r4["r4_r3_autonomous"].sum())
r3_wrong = int(r4["r4_r3_wrong"].sum())
r3_correct = r3_auto - r3_wrong

added = int(r4["r4_uncertainty_escalated"].sum())
captured = int(
    (
        r4["r4_uncertainty_escalated"] &
        r4["r4_r3_wrong"]
    ).sum()
)

correct_escalated = added - captured

r4_esc = int(r4["r4_final_escalated"].sum())
r4_auto = int(r4["r4_final_autonomous"].sum())
r4_wrong = int(r4["r4_final_wrong"].sum())
r4_correct = int(r4["r4_final_correct"].sum())

print("\n" + "=" * 110)
print("RECONSTRUCTION CHECK")
print("=" * 110)

print("R3 escalated             :", r3_esc)
print("R3 autonomous            :", r3_auto)
print("R3 autonomous correct    :", r3_correct)
print("R3 autonomous errors     :", r3_wrong)

print("\nAdded R4 escalations     :", added)
print("R3 errors captured       :", captured)
print("Correct cases escalated  :", correct_escalated)

print("\nR4 escalated             :", r4_esc)
print("R4 autonomous            :", r4_auto)
print("R4 autonomous correct    :", r4_correct)
print("R4 autonomous errors     :", r4_wrong)

# Exact expected completed-run counts
assert r3_esc == 19
assert r3_auto == 341
assert r3_correct == 329
assert r3_wrong == 12

assert added == 35
assert captured == 7
assert correct_escalated == 28

assert r4_esc == 54
assert r4_auto == 306
assert r4_correct == 301
assert r4_wrong == 5

print("\nPASS — recovered calculations reproduce completed R4 counts exactly")

# ----------------------------------------------------------------------------------------------------
# 5. System comparison
# ----------------------------------------------------------------------------------------------------

comparison = pd.DataFrame([
    {
        "system": "Frozen R3",
        "total_n": n,
        "autonomous_n": r3_auto,
        "escalated_n": r3_esc,
        "coverage": r3_auto / n,
        "escalation_rate": r3_esc / n,
        "autonomous_correct": r3_correct,
        "autonomous_errors": r3_wrong,
        "autonomous_accuracy": r3_correct / r3_auto,
        "selective_risk": r3_wrong / r3_auto,
        "correct_autonomous_over_total": r3_correct / n,
    },
    {
        "system": "Frozen R4",
        "total_n": n,
        "autonomous_n": r4_auto,
        "escalated_n": r4_esc,
        "coverage": r4_auto / n,
        "escalation_rate": r4_esc / n,
        "autonomous_correct": r4_correct,
        "autonomous_errors": r4_wrong,
        "autonomous_accuracy": r4_correct / r4_auto,
        "selective_risk": r4_wrong / r4_auto,
        "correct_autonomous_over_total": r4_correct / n,
    }
])

comparison.to_csv(
    OUT / "R4_RECOVERED_final_system_comparison.csv",
    index=False
)

print("\nSYSTEM COMPARISON")
print("-" * 110)
print(comparison.to_string(index=False))

# ----------------------------------------------------------------------------------------------------
# 6. UC-kind error capture
# ----------------------------------------------------------------------------------------------------

errors = r4[r4["r4_r3_wrong"]].copy()

uc_capture = (
    errors.groupby("uc_kind")
    .agg(
        r3_errors=("id", "size"),
        captured_by_r4=("r4_uncertainty_escalated", "sum")
    )
    .reset_index()
)

uc_capture["captured_by_r4"] = (
    uc_capture["captured_by_r4"].astype(int)
)

uc_capture["missed_by_r4"] = (
    uc_capture["r3_errors"] -
    uc_capture["captured_by_r4"]
)

uc_capture["capture_rate"] = (
    uc_capture["captured_by_r4"] /
    uc_capture["r3_errors"]
)

uc_capture.to_csv(
    OUT / "R4_RECOVERED_error_capture_by_uc_kind.csv",
    index=False
)

print("\nERROR CAPTURE BY UC KIND")
print("-" * 110)
print(uc_capture.to_string(index=False))

# Known completed-run result
generic = uc_capture[uc_capture["uc_kind"] == "generic"].iloc[0]
unseen = uc_capture[uc_capture["uc_kind"] == "unseen"].iloc[0]

assert int(generic["r3_errors"]) == 5
assert int(generic["captured_by_r4"]) == 4

assert int(unseen["r3_errors"]) == 7
assert int(unseen["captured_by_r4"]) == 3

# ----------------------------------------------------------------------------------------------------
# 7. Save reconstructed decisions
# ----------------------------------------------------------------------------------------------------

decision_path = (
    OUT /
    "R4_RECOVERED_frozen_policy_heldout_decisions.csv"
)

r4.to_csv(
    decision_path,
    index=False
)

# ----------------------------------------------------------------------------------------------------
# 8. Recovery metadata — clearly distinguish originals from recovered files
# ----------------------------------------------------------------------------------------------------

summary = {
    "status": "RECOVERED_FROM_COMPLETED_FROZEN_RUN_AFTER_COLAB_RESET",

    "methodological_note":
        "R4 was completed before the Colab runtime reset. "
        "This file reconstructs the completed frozen-policy metrics "
        "from the SHA-verified frozen R3 evidence and the previously "
        "recorded immutable R4 threshold. No threshold tuning or "
        "heldout-driven policy modification was performed during recovery.",

    "frozen_r3_tau": R3_TAU,
    "frozen_r4_margin_threshold": R4_MARGIN_TAU,

    "original_completed_run_hashes": {
        "r4_policy_sha256": ORIGINAL_R4_POLICY_SHA,
        "r4_heldout_decisions_sha256": ORIGINAL_R4_HELDOUT_DECISIONS_SHA,
        "r4_heldout_summary_sha256": ORIGINAL_R4_HELDOUT_SUMMARY_SHA,
    },

    "r3": {
        "total": n,
        "escalated": r3_esc,
        "autonomous": r3_auto,
        "autonomous_correct": r3_correct,
        "autonomous_errors": r3_wrong,
        "coverage": r3_auto / n,
        "autonomous_accuracy": r3_correct / r3_auto,
        "selective_risk": r3_wrong / r3_auto,
    },

    "r4": {
        "added_uncertainty_escalations": added,
        "errors_captured": captured,
        "correct_cases_escalated": correct_escalated,
        "total_escalated": r4_esc,
        "autonomous": r4_auto,
        "autonomous_correct": r4_correct,
        "autonomous_errors": r4_wrong,
        "coverage": r4_auto / n,
        "autonomous_accuracy": r4_correct / r4_auto,
        "selective_risk": r4_wrong / r4_auto,
        "r3_residual_error_capture": captured / r3_wrong,
        "added_escalation_precision": captured / added,
    }
}

summary_path = (
    OUT /
    "R4_RECOVERED_completed_run_summary.json"
)

with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

# ----------------------------------------------------------------------------------------------------
# 9. Recovery provenance record
# ----------------------------------------------------------------------------------------------------

provenance = {
    "recovery_status":
        "POST_RUNTIME_RESET_RECOVERY",

    "r3_source_zip":
        "XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip",

    "r3_source_zip_sha256":
        "ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152",

    "r3_internal_manifest_entries_verified":
        29,

    "r4_experiment_was_completed_before_reset":
        True,

    "r4_reexecuted_during_recovery":
        False,

    "r4_retuned_during_recovery":
        False,

    "original_r4_policy_sha256":
        ORIGINAL_R4_POLICY_SHA,

    "original_r4_heldout_decisions_sha256":
        ORIGINAL_R4_HELDOUT_DECISIONS_SHA,

    "original_r4_heldout_summary_sha256":
        ORIGINAL_R4_HELDOUT_SUMMARY_SHA,
}

with open(
    OUT / "R4_RECOVERY_PROVENANCE.json",
    "w"
) as f:
    json.dump(provenance, f, indent=2)

# ----------------------------------------------------------------------------------------------------
# 10. SHA256 recovered artifacts
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("RECOVERED ARTIFACT HASHES")
print("=" * 110)

for p in sorted(OUT.glob("R4_RECOVERED_*")):

    sha = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    print(p.name)
    print(" SHA256:", sha)

print("\n" + "=" * 110)
print("PASS — ESSENTIAL R4 EVIDENCE RECOVERED")
print("PASS — COMPLETED R4 COUNTS REPRODUCED EXACTLY")
print("PASS — NO R4 RETUNING OR RE-EXPERIMENTATION PERFORMED")
print("=" * 110)

R4-RECOVERY-03 — RECONSTRUCT ESSENTIAL R4 EVIDENCE

Frozen R3 heldout rows: 360

RECONSTRUCTION CHECK
R3 escalated             : 19
R3 autonomous            : 341
R3 autonomous correct    : 329
R3 autonomous errors     : 12

Added R4 escalations     : 35
R3 errors captured       : 7
Correct cases escalated  : 28

R4 escalated             : 54
R4 autonomous            : 306
R4 autonomous correct    : 301
R4 autonomous errors     : 5

PASS — recovered calculations reproduce completed R4 counts exactly

SYSTEM COMPARISON
--------------------------------------------------------------------------------------------------------------
   system  total_n  autonomous_n  escalated_n  coverage  escalation_rate  autonomous_correct  autonomous_errors  autonomous_accuracy  selective_risk  correct_autonomous_over_total
Frozen R3      360           341           19  0.947222         0.052778                 329                 12             0.964809        0.035191                       0.913889
Froze

In [7]:
# ====================================================================================================
# R4-21 — CREATE FINAL R4 FINDINGS AND INTERPRETATION DOCUMENT
# ====================================================================================================

from pathlib import Path

OUT = Path("/content/R4_work/R4_results")
OUT.mkdir(parents=True, exist_ok=True)

REPORT = OUT / "R4_FINAL_FINDINGS_AND_INTERPRETATION.md"

text = r"""# R4 — Uncertainty-Aware Selective Escalation

## 1. Objective

R4 evaluates whether uncertainty information from the frozen R3 selection
pipeline can be used to selectively escalate unreliable autonomous decisions.

The research question is:

> Can uncertainty-aware selective escalation reduce residual autonomous errors
> while preserving a useful level of autonomous coverage?

R4 is evaluated as an extension of the frozen R3 candidate-masking system.

---

## 2. Experimental Integrity

R4 was designed to preserve strict separation between development/calibration
data and held-out evaluation data.

- Development/calibration observations: 840
- Frozen held-out observations: 360
- Frozen R3 feasible-mass threshold: 0.5486315987262682
- R4 uncertainty signal: top-1 minus top-2 probability margin
- Lower margin indicates greater uncertainty.
- R4 threshold selection was performed using development/OOF evidence only.
- Held-out labels were not used for threshold selection.
- The R4 policy was frozen before held-out evaluation.
- No post-hoc threshold modification was performed after observing held-out results.

Original frozen R4 policy SHA256:

1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42

Original frozen held-out decision SHA256:

8f7aab8ed6cdf66a5616d750f070ea02614c0fd1f3e0bf043f36d07f7527611a

Original frozen held-out summary SHA256:

9c4a1c98cf2d17da30045f60a5021b64bdd6deb410f3d7a310b207dff19ad412

---

## 3. Development-Set Uncertainty Analysis

The R4 development analysis examined several uncertainty signals:

1. 1 - top-1 probability
2. Predictive entropy
3. 1 - top-1/top-2 probability margin
4. 1 - feasible probability mass

Their development error-detection AUC values were:

| Signal | Error-detection AUC |
|---|---:|
| 1 - top-1 | 0.903454 |
| Entropy | 0.902956 |
| 1 - margin | 0.902169 |
| 1 - feasible mass | 0.765217 |

This showed that confidence-, entropy-, and margin-based uncertainty signals
were substantially more informative for detecting residual errors than feasible
probability mass alone.

At approximately 10% additional escalation:

- top-1 captured 35 of 62 development errors;
- margin captured 36 of 62;
- entropy captured 36 of 62;
- feasible mass captured 28 of 62.

---

## 4. Frozen R4 Operating Point

The selected R4 policy uses the top-1/top-2 probability margin.

The frozen margin threshold is:

0.7097637192002952

The rule is:

1. Apply the already-frozen R3 policy.
2. If R3 escalates, retain the escalation.
3. If R3 would automate, calculate the top-1 minus top-2 probability margin.
4. If margin < 0.7097637192002952, escalate the decision.
5. Otherwise retain autonomous execution.

The threshold was frozen from OOF development evidence before evaluating the
360-row held-out set.

Development performance at this operating point:

- Total development observations: 840
- Combined escalations: 136
- Autonomous observations: 704
- Coverage: 0.838095
- Autonomous errors: 13
- Autonomous accuracy: 0.981534

---

## 5. One-Shot Held-Out Evaluation

The frozen policy was applied once to the 360-row held-out evaluation set.

### Frozen R3 baseline

- Total observations: 360
- Escalated: 19
- Autonomous: 341
- Coverage: 0.947222
- Escalation rate: 0.052778
- Autonomous correct: 329
- Autonomous errors: 12
- Autonomous accuracy: 0.964809
- Selective risk: 0.035191
- Correct autonomous / total: 0.913889

### Frozen R4

- Total observations: 360
- Additional uncertainty escalations: 35
- Total escalated: 54
- Autonomous: 306
- Coverage: 0.850000
- Escalation rate: 0.150000
- Autonomous correct: 301
- Autonomous errors: 5
- Autonomous accuracy: 0.983660
- Selective risk: 0.016340
- Correct autonomous / total: 0.836111

---

## 6. R4 Improvement Over R3

R4 reduced the residual autonomous errors from:

12 -> 5

Therefore, 7 of the 12 residual R3 errors were successfully intercepted by
uncertainty-aware escalation.

Residual error capture:

7 / 12 = 58.33%

Autonomous accuracy improved from:

96.48% -> 98.37%

Selective risk decreased from:

3.52% -> 1.63%

Relative selective-risk reduction:

53.57%

However, this reliability improvement required a reduction in autonomous
coverage:

94.72% -> 85.00%

Coverage change:

-9.72 percentage points

Therefore, R4 should be interpreted as improving the safety/reliability of
autonomous operation rather than increasing autonomous throughput.

---

## 7. Escalation Efficiency

R4 added 35 uncertainty-driven escalations.

Among these:

- Residual R3 errors captured: 7
- Correct R3 decisions also escalated: 28

Added escalation precision:

7 / 35 = 20%

This demonstrates an important trade-off.

The uncertainty mechanism successfully removes more than half of the remaining
R3 errors, but it also escalates some correct cases.

Thus R4 implements conservative selective autonomy rather than perfect
error identification.

---

## 8. Error Capture by Generalization Category

Residual R3 errors consisted of:

### Generic cases

- R3 errors: 5
- Captured by R4: 4
- Missed by R4: 1
- Capture rate: 80%

### Unseen cases

- R3 errors: 7
- Captured by R4: 3
- Missed by R4: 4
- Capture rate: 42.86%

The uncertainty mechanism is therefore substantially more successful on generic
residual errors than on unseen residual errors.

---

## 9. High-Confidence Error Limitation

The R4 audit identified an important limitation.

Some residual errors were made with high model confidence and large
top-1/top-2 probability margins.

Captured errors had substantially lower margins, while the five missed errors
had high margins.

Approximate mean profiles observed during the completed R4 audit were:

### Captured residual errors

- n = 7
- mean top-1 confidence = 0.646939
- mean margin = 0.360555
- mean normalized entropy = 0.703867
- mean feasible mass = 0.957960

### Missed residual errors

- n = 5
- mean top-1 confidence = 0.945313
- mean margin = 0.908073
- mean normalized entropy = 0.209493
- mean feasible mass = 0.991204

This means conventional uncertainty measures cannot detect every model error.

In particular, confidently wrong predictions remain a significant challenge.

This limitation must not be addressed by post-hoc threshold tuning on the
held-out set because doing so would compromise evaluation integrity.

Instead, high-confidence errors motivate future work involving richer
distribution-shift detection, semantic consistency checks, ensemble
disagreement, calibrated out-of-distribution detection, or additional
verification mechanisms.

---

## 10. Main Finding

The central R4 finding is:

> A frozen uncertainty-aware selective escalation layer substantially improves
> the reliability of the R3 autonomous network-slicing decision pipeline,
> increasing autonomous accuracy from 96.48% to 98.37% and reducing selective
> risk by approximately 53.57%, while reducing autonomous coverage from
> 94.72% to 85.00%.

This represents a deliberate reliability-versus-coverage trade-off.

---

## 11. Research Interpretation

R3 addressed feasibility-aware candidate masking.

R4 adds a second trust mechanism:

- R3 asks whether the candidate decision is feasible.
- R4 additionally asks whether the model is sufficiently certain about the
  autonomous decision.

The resulting architecture therefore separates:

1. feasibility-based safety filtering; and
2. uncertainty-based selective autonomy.

This provides a stronger trust architecture than either mechanism alone.

---

## 12. Limitations

The following limitations should be explicitly reported.

1. R4 improves reliability at the cost of autonomous coverage.
2. Only 20% of additional R4 escalations corresponded to residual R3 errors.
3. Some correct decisions were conservatively escalated.
4. High-confidence incorrect predictions cannot reliably be detected using
   probability margin alone.
5. Unseen-case error capture (42.86%) was lower than generic-case error
   capture (80%).
6. The current experiment evaluates selective escalation rather than a complete
   production-scale distribution-shift detection framework.
7. Additional robustness testing should be treated as separate future
   experimentation rather than post-hoc modification of the frozen R4 result.

---

## 13. Runtime-Recovery Provenance

After completion of the R4 experiment, the Google Colab runtime reset and
removed the temporary /content workspace.

The experiment itself was not rerun.

The frozen R3 evidence package was subsequently restored from:

XAgentic_R3_Candidate_Masking_FINAL_2026-10-07.zip

with SHA256:

ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152

All 29 entries in the internal R3 SHA256 manifest were successfully verified.

The recovered evidence contained:

- 840 OOF development/calibration observations;
- 360 frozen held-out observations; and
- 360 frozen R3 held-out decisions.

The already-frozen R4 margin threshold was then reapplied to the SHA-verified
R3 held-out evidence.

The reconstruction reproduced the completed R4 counts exactly:

- R3 autonomous: 341
- R3 errors: 12
- R4 added escalations: 35
- R4 errors captured: 7
- R4 autonomous: 306
- R4 autonomous correct: 301
- R4 autonomous errors: 5

No R4 threshold was retuned and no held-out-driven policy modification was
performed during recovery.

Recovered files are explicitly named R4_RECOVERED_* to distinguish them from
the original byte-level artifacts.

---

## 14. Conclusion

R4 demonstrates that uncertainty-aware selective escalation can materially
improve the reliability of an already feasibility-aware autonomous
network-slicing pipeline.

Compared with frozen R3, the R4 mechanism reduced autonomous errors from 12 to
5 and reduced selective risk by approximately 53.57%.

The improvement comes with a measurable coverage cost: autonomous coverage
decreased from 94.72% to 85.00%.

Consequently, R4 supports the use of selective autonomy for trustworthy
agentic network-slicing systems where reliability is prioritized over maximum
automation.

The remaining high-confidence errors provide a clear motivation for subsequent
experiments focused on stronger verification, robustness, and detection of
confidently incorrect decisions.
"""

REPORT.write_text(text, encoding="utf-8")

print("=" * 110)
print("R4-21 — FINAL R4 WRITE-UP")
print("=" * 110)
print("Written:", REPORT)
print("Bytes  :", f"{REPORT.stat().st_size:,}")
print("\nPASS — R4 final findings document created")

R4-21 — FINAL R4 WRITE-UP
Written: /content/R4_work/R4_results/R4_FINAL_FINDINGS_AND_INTERPRETATION.md
Bytes  : 10,176

PASS — R4 final findings document created


In [8]:
# ====================================================================================================
# R4-22 — FINAL R4 AUDIT, SHA256 MANIFEST, AND EVIDENCE PACKAGE
# ====================================================================================================

from pathlib import Path
import hashlib
import pandas as pd
import json
import zipfile

print("=" * 110)
print("R4-22 — FINAL R4 AUDIT, MANIFEST, AND EVIDENCE PACKAGE")
print("=" * 110)

WORK = Path("/content/R4_work")
OUT = WORK / "R4_results"

ZIP_OUT = Path(
    "/content/XAgentic_R4_Uncertainty_Escalation_FINAL_2026-10-07.zip"
)

# ----------------------------------------------------------------------------------------------------
# Required recovered R4 evidence
# ----------------------------------------------------------------------------------------------------

required = [
    "R4_RECOVERED_completed_run_summary.json",
    "R4_RECOVERED_error_capture_by_uc_kind.csv",
    "R4_RECOVERED_final_system_comparison.csv",
    "R4_RECOVERED_frozen_policy_heldout_decisions.csv",
    "R4_RECOVERY_PROVENANCE.json",
    "R4_recovered_R3_manifest_audit.csv",
    "R4_runtime_recovery_summary.csv",
    "R4_FINAL_FINDINGS_AND_INTERPRETATION.md",
]

print("\nREQUIRED ARTIFACT CHECK")
print("-" * 110)

for name in required:
    p = OUT / name
    assert p.exists(), f"STOP — missing {name}"
    print("PASS", name)

# ----------------------------------------------------------------------------------------------------
# Verify summary
# ----------------------------------------------------------------------------------------------------

summary = json.loads(
    (OUT / "R4_RECOVERED_completed_run_summary.json").read_text()
)

assert summary["r3"]["autonomous"] == 341
assert summary["r3"]["autonomous_errors"] == 12

assert summary["r4"]["autonomous"] == 306
assert summary["r4"]["autonomous_correct"] == 301
assert summary["r4"]["autonomous_errors"] == 5
assert summary["r4"]["errors_captured"] == 7

print("\nPASS — frozen R4 metrics verified")

# ----------------------------------------------------------------------------------------------------
# Preserve original completed-run identifiers
# ----------------------------------------------------------------------------------------------------

original_hashes = {
    "original_R4_policy_sha256":
        "1b2d69794b1508f807227b12e78fdabce857683731cb91ee1879f7069029db42",

    "original_R4_heldout_decisions_sha256":
        "8f7aab8ed6cdf66a5616d750f070ea02614c0fd1f3e0bf043f36d07f7527611a",

    "original_R4_heldout_summary_sha256":
        "9c4a1c98cf2d17da30045f60a5021b64bdd6deb410f3d7a310b207dff19ad412",

    "frozen_R3_source_zip_sha256":
        "ba0f680c372ddda498eba2f8b84f5df9a599dc04810a679b53ce77af6fce0152"
}

with open(
    OUT / "R4_ORIGINAL_COMPLETED_RUN_HASHES.json",
    "w"
) as f:
    json.dump(original_hashes, f, indent=2)

# ----------------------------------------------------------------------------------------------------
# Build inventory and SHA manifest
# ----------------------------------------------------------------------------------------------------

files = sorted(
    p for p in OUT.rglob("*")
    if p.is_file()
    and p.name not in {
        "R4_FINAL_ARTIFACT_INVENTORY.csv",
        "R4_FINAL_SHA256_MANIFEST.csv"
    }
)

inventory_rows = []
manifest_rows = []

for p in files:

    rel = p.relative_to(OUT).as_posix()

    sha = hashlib.sha256(
        p.read_bytes()
    ).hexdigest()

    inventory_rows.append({
        "relative_path": rel,
        "size_bytes": p.stat().st_size
    })

    manifest_rows.append({
        "relative_path": rel,
        "size_bytes": p.stat().st_size,
        "sha256": sha
    })

inventory = pd.DataFrame(inventory_rows)
manifest = pd.DataFrame(manifest_rows)

inventory_path = OUT / "R4_FINAL_ARTIFACT_INVENTORY.csv"
manifest_path = OUT / "R4_FINAL_SHA256_MANIFEST.csv"

inventory.to_csv(inventory_path, index=False)
manifest.to_csv(manifest_path, index=False)

print("\nArtifacts inventoried:", len(inventory))
print("Manifest entries     :", len(manifest))

# ----------------------------------------------------------------------------------------------------
# Create ZIP
# ----------------------------------------------------------------------------------------------------

if ZIP_OUT.exists():
    ZIP_OUT.unlink()

with zipfile.ZipFile(
    ZIP_OUT,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as z:

    for p in sorted(OUT.rglob("*")):

        if p.is_file():

            arcname = (
                Path("XAgentic_R4_Uncertainty_Escalation_FINAL")
                / p.relative_to(OUT)
            )

            z.write(
                p,
                arcname.as_posix()
            )

# ----------------------------------------------------------------------------------------------------
# Verify ZIP
# ----------------------------------------------------------------------------------------------------

with zipfile.ZipFile(ZIP_OUT, "r") as z:

    bad = z.testzip()

    assert bad is None, (
        f"STOP — ZIP integrity failure: {bad}"
    )

    members = z.namelist()

zip_sha = hashlib.sha256(
    ZIP_OUT.read_bytes()
).hexdigest()

print("\nFINAL PACKAGE")
print("-" * 110)

print("ZIP file   :", ZIP_OUT)
print("ZIP members:", len(members))
print("ZIP bytes  :", f"{ZIP_OUT.stat().st_size:,}")
print("ZIP SHA256 :", zip_sha)

# ----------------------------------------------------------------------------------------------------
# Final closure summary
# ----------------------------------------------------------------------------------------------------

print("\n" + "=" * 110)
print("R4 FINAL RESULT")
print("=" * 110)

print("""
Frozen R3:
  Coverage             : 94.72%
  Autonomous accuracy  : 96.48%
  Autonomous errors    : 12
  Selective risk       : 3.52%

Frozen R4:
  Coverage             : 85.00%
  Autonomous accuracy  : 98.37%
  Autonomous errors    : 5
  Selective risk       : 1.63%

R4 captured:
  7 / 12 residual R3 errors = 58.33%

Relative selective-risk reduction:
  53.57%

Interpretation:
  R4 successfully improves reliability through selective escalation,
  with an explicit reduction in autonomous coverage.
""")

print("=" * 110)
print("PASS — FINAL R4 EVIDENCE PACKAGE CREATED")
print("PASS — ZIP INTEGRITY VERIFIED")
print("PASS — R4 CAN NOW BE CLOSED")
print("=" * 110)

R4-22 — FINAL R4 AUDIT, MANIFEST, AND EVIDENCE PACKAGE

REQUIRED ARTIFACT CHECK
--------------------------------------------------------------------------------------------------------------
PASS R4_RECOVERED_completed_run_summary.json
PASS R4_RECOVERED_error_capture_by_uc_kind.csv
PASS R4_RECOVERED_final_system_comparison.csv
PASS R4_RECOVERED_frozen_policy_heldout_decisions.csv
PASS R4_RECOVERY_PROVENANCE.json
PASS R4_recovered_R3_manifest_audit.csv
PASS R4_runtime_recovery_summary.csv
PASS R4_FINAL_FINDINGS_AND_INTERPRETATION.md

PASS — frozen R4 metrics verified

Artifacts inventoried: 9
Manifest entries     : 9

FINAL PACKAGE
--------------------------------------------------------------------------------------------------------------
ZIP file   : /content/XAgentic_R4_Uncertainty_Escalation_FINAL_2026-10-07.zip
ZIP members: 11
ZIP bytes  : 33,949
ZIP SHA256 : 101d81ca812aec3af359c5d00074705d0b9785b3bfb3d87e02874695bd138677

R4 FINAL RESULT

Frozen R3:
  Coverage             : 94.7

In [9]:
from google.colab import files
from pathlib import Path
import hashlib

ZIP = Path("/content/XAgentic_R4_Uncertainty_Escalation_FINAL_2026-10-07.zip")
REPORT = Path("/content/R4_work/R4_results/R4_FINAL_FINDINGS_AND_INTERPRETATION.md")

EXPECTED_SHA = "101d81ca812aec3af359c5d00074705d0b9785b3bfb3d87e02874695bd138677"

assert ZIP.exists()
assert REPORT.exists()

actual = hashlib.sha256(ZIP.read_bytes()).hexdigest()

print("FINAL PRE-DOWNLOAD CHECK")
print("=" * 90)
print("ZIP       :", ZIP.name)
print("Bytes     :", f"{ZIP.stat().st_size:,}")
print("Expected  :", EXPECTED_SHA)
print("Actual    :", actual)

assert actual == EXPECTED_SHA

print("\nPASS — final R4 ZIP verified")
print("\nDownloading final R4 package...")
files.download(str(ZIP))

FINAL PRE-DOWNLOAD CHECK
ZIP       : XAgentic_R4_Uncertainty_Escalation_FINAL_2026-10-07.zip
Bytes     : 33,949
Expected  : 101d81ca812aec3af359c5d00074705d0b9785b3bfb3d87e02874695bd138677
Actual    : 101d81ca812aec3af359c5d00074705d0b9785b3bfb3d87e02874695bd138677

PASS — final R4 ZIP verified



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
from google.colab import files

files.download(
    "/content/R4_work/R4_results/R4_FINAL_FINDINGS_AND_INTERPRETATION.md"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>